# TGNN 9클래스 통합본 — 팀장님 피드백 반영판

손은총 · 자금세탁소 · 2026-09-29 · HI-Large · 상태 **제안·미결**(결정은 팀 다섯 명이 함께 한다)

- **한 파일로 전부 돈다**: DuckDB 연결 → 36피처 → 하루 캐시 → 이웃 샘플러 → 모델 → 학습 → 채점 → 자체 검사.
  - §1~§7 은 태훈님 `daily_pipeline.py` 를 옮긴 원본 코드다(원본과 점수 차이 0 을 §10 에서 확인).
  - §7-수정 · §8~§10 이 고친 부분이다. 무엇을 고칠지는 스위치(팔)로 켠다.
- **지금 돌고 있는 코드와 계산이 같다.** 서버에서 도는 `TGNN9클래스_수정판_한파일_20260929.ipynb` 에서 바꾼 것은 이 머리말, 기본 팔(`shuffle_m`), 출력 폴더(`_team/`), 맨 끝 API 순서 함수뿐이다.
- 값은 **val 로만** 비교한다. test 는 팀이 최종 설정을 고른 뒤 한 번만 연다.

## 1. 팀장님 피드백 대응표

| 피드백 | 반영 | 스위치 · 팔 | 상태(09-29 08:4x UTC) |
|---|---|---|---|
| 배치 셔플(시드 고정) | **반영** — 태훈님 식 그대로 `default_rng(seed + epoch + dayofyear)`. 하루 안 학습 행만 섞는다. "무작위 금지" 규칙의 예외로 승인했다 | `shuffle` · `shuffle_m` | **진행 중 — 효과가 크다**(아래 결과) |
| 라벨을 팀 결정에 맞추기(패턴 밖 세탁 → 클래스 0) | **반영** — 패턴 밖 세탁을 정상 클래스로 학습·평가에 넣는다. train 43,095행은 CSSMC 와 따로 전부(가중치 1), val +16,691 · test +17,063행 | `label_merge` · `_m` 팔 전부 | 반영됨 |
| 샘플링 방향(out-edge 넣기) | **반영** — 계좌마다 보낸 거래 최근 10건 문맥을 새로 만들고, 양방향 2홉 샘플러로 모은다 | `reverse_mp` · `revmp_m` | 대기(나가는 문맥 캐시 먼저, 약 4시간) |
| Reverse MP(별도 가중치) | **반영** — 층마다 받은 쪽 conv + 뒤집은 엣지 전용 conv(`conv1r·conv2r`, 가중치 따로) 합산 | `reverse_mp` · `revmp_m` | 대기 |
| Ego ID | **반영** — 노드 입력 1열. 단 **배치에 든 분류 대상 4,096건 전부의 양 끝 계좌**에 1 을 붙인다(거래마다 따로가 아님). 그래서 `_m` 팔은 채점도 섞은 배치로 한다 | `ego` · `revmp_ego_m` | 대기 |
| Port numbering | **미반영** — 우리 문맥은 계좌당 최근 10건이라 "첫 거래 시각 순 번호"의 정의가 문헌과 달라진다 | — | 보류(설계 필요) |
| 이웃 10 → 64~100 | **미반영** — 하루마다 30일 문맥을 저장하는 지금 캐시는 10건으로도 102 GB 다. 100건이면 수백 GB 라 샘플러를 새로 짜야 한다. 또 어텐션은 가중 평균이라 이웃이 많아도 수를 못 센다(합 집계 필요). 받은 상대 수는 노드 피처(`log_in_neighbors` 등)에 이미 있다 | — | 보류(revmp·ego 결과 뒤 팀 결정) |
| 클래스 순서를 API 에 맞추기 | **함수만** — §11 `to_api_order`(이름으로 매핑). `scores.parquet` 형식은 API 명세를 확인한 뒤 쓴다. G-SCATTER = GATHER-SCATTER, S-GATHER = SCATTER-GATHER 로 봤다(팀 확인 필요) | §11 | 내보낼 때 |
| 시드 3개(42·0·1) | 최종 설정에만. **주의**: 셔플 시드 식이 정수 합이라 시드 0 의 epoch e+1 과 시드 1 의 epoch e 가 같은 순서다. 시드는 15 이상 띄운다(예: 42·0·100) | `FIX_SEED` | 최종 설정 뒤 |
| §2 0.9996 인용 금지 · 트리/GNN 입력 차이 | 인용하지 않는다. 입력 차이는 **36피처 판에서 없어졌다** — TGNN(노드 19 × 2 + 엣지 17)과 LightGBM(55열)이 같은 원본 열을 쓴다 | — | 해소 |

- `reg`(dropout·weight decay)는 뺐다. 진단에서 train 유형 정답률조차 0.11~0.13(과소적합)이었다.
- `interleave`(규칙적 섞기)·`lossbal`(고정 분모)도 연쇄에서 뺐다.
  - 셔플과 같은 문제를 겨눈다.
  - 셔플 뒤에는 유형별 실효 가중치가 8종 모두 0.98~1.02 라 lossbal 이 고칠 것이 남지 않는다.

## 2. 지금까지 결과 (val · 09-29 08:4x UTC · 시드 42 한 번 · 진행 중)

| 설정 | 라벨 | epoch | macro-AP | 바닥 | **바닥대비** | 세탁 여부 AP | **세탁 행 8종 정답률** |
|---|---|---:|---:|---:|---:|---:|---:|
| 원본 TGNN36(시간순 배치) | 제거 | 3(최고) | 0.183 | 0.209 | −0.026 | 0.881 | 0.104 |
| `norm`(입력 표준화 + LayerNorm) | 제거 | 4(최고) | 0.200 | 0.213 | −0.013 | 0.914 | 0.164 |
| **`shuffle_m`(병합 + 시드 셔플)** | 병합 | 1 | **0.285** | 0.209 | **+0.077** | 0.880 | 0.394 |
| 〃 | 병합 | 2 | 0.253 | 0.210 | +0.044 | 0.886 | 0.410 |
| 〃 | 병합 | 3 | 0.248 | 0.199 | +0.049 | 0.791 | **0.417** |
| 참고: LightGBM 55열(r299) | 제거 | — | 0.308 | 0.219 | +0.090 | 0.969 | 0.385 |

- **바닥** = (정상 AP + 세탁 여부 AP) ÷ 9 다. 유형을 전혀 못 가리고 사전확률만 곱했을 때의 macro-AP 다.
  - 바닥대비가 0 보다 커야 유형 머리가 쓸모 있다.
  - 원본 TGNN 은 바닥 아래였다. 유형 머리가 오히려 점수를 깎았다.
- **세탁 행 8종 정답률** = 정답이 패턴인 행에서 8종 중 가장 높은 것을 고른 정답률이다. 병합과 무관하다.
- **원인 진단**(`보고서_TGNN_유형붕괴_원인_손은총_20260929.md`): 시간순 배치 때문에 유형을 train 에서도 못 배웠다.
  - 배치 29% 가 세탁 0건, 52% 가 세탁 1건 이하였다.
  - 배치 평균 손실이 유형별 가중치를 0.27~1.77배로 비틀었다(BIPARTITE 0.34배, STACK 0.27배 → 거의 예측 안 함).
  - 셔플로 풀렸다. train 세탁 행의 조건부 유형 CE 는 원본 체크포인트(ep1~6)에서 2.20~2.22 였다. shuffle_m 은 학습 중 평균 1.61 → 1.49 → 1.47(ep1~3)로, 사전분포 엔트로피 2.03 아래다. 측정 방식이 조금 다르다: 원본은 체크포인트로 다시 계산했고, shuffle_m 은 학습 중 순전파 값이다.
  - **팀장님 지적대로 배치 순서가 원인이었다.**
- 아직 판단하지 않은 것
  - epoch 이 지나며 유형 정답률은 오르는데, 세탁 여부 AP 와 macro-AP 는 내려간다.
  - 평가 행의 약 65% 는 학습 때 본 시도(attempt)다. 처음 보는 시도만 뗀 정답률을 따로 계산할 예정이다. 비교 기준은 T9 GNN 0.286, LightGBM 0.262 다.

## 3. 실행 방법

환경 변수로 팔과 단계를 고른다. 아무것도 안 주면 **`shuffle_m` 팔의 가벼운 자체 검사(§10, CPU 몇 분)만** 돈다.

```bash
cd /workspace/Final_preprocess/HI-Large_EDA/tgnn36_20260928
NB=TGNN9클래스_통합본_팀장전달_20260929.ipynb
# 1) 스모크(2일 · 1 epoch) → 2) 본 학습 · 채점 (GPU, 팔마다 약 1.5~3시간)
FIX_ARM=shuffle_m FIX_SMOKE=1 FIX_RUN_SELFCHECK=0 FIX_RUN_TRAIN=1 FIX_RUN_EVAL=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 $NB --output-dir _team/shuffle_m_smoke --output run.ipynb
FIX_ARM=shuffle_m FIX_RUN_SELFCHECK=0 FIX_RUN_TRAIN=1 FIX_RUN_EVAL=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 $NB --output-dir _team/shuffle_m --output run.ipynb
# revmp_m · revmp_ego_m 앞에 한 번만: 나가는 문맥 캐시(DuckDB, 반드시 단독 실행, 약 4시간 추정)
FIX_ARM=off FIX_RUN_OUT_CACHE=1 FIX_RUN_SELFCHECK=0 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 $NB --output-dir _team/common --output out_cache.ipynb
```

| 변수 | 뜻 |
|---|---|
| `FIX_ARM` | 팔. `shuffle_m`(병합+셔플) → `revmp_m`(+양방향 · Reverse MP) → `revmp_ego_m`(+ego) 순으로 쌓는다. 앞 팔들(`off`·`norm`·`lossbal`·`interleave`·`reg`·`revmp`·`revmp_ego`)은 병합 전 라벨의 한 가지씩 바꾼 팔 |
| `FIX_SEED` | 시드(기본 42). 42 가 아니면 출력 폴더에 `_s<시드>` |
| `FIX_SMOKE` | 1 이면 train 2일 · val 1일 · test 1일 · 1 epoch |
| `FIX_RUN_TRAIN` · `FIX_RUN_EVAL` · `FIX_RUN_OUT_CACHE` · `FIX_RUN_SELFCHECK` | 단계 켜기 |

- 결과: `_team/<팔>/results/학습기록.csv`(epoch 마다 val macro-AP · 바닥 · 바닥대비 · 조건부 8종 · 학습 중 유형 CE)
  - `지표.csv`(고른 epoch 의 val·test), 체크포인트 `_team/<팔>/models/ep*.pt`
- 서버에서는 같은 계산이 `_fix/<팔>/` 에 이미 돌고 있다(`run_fix_arms.sh` 가 팔마다 스모크 → 본 학습).
- 안전장치
  - 다른 무거운 작업이 돌면 시작하지 않는다.
  - 메모리 anon 18.5 GiB 를 넘으면 체크포인트를 남기고 멈춘다. 서버 mem_guard 는 19.14 GiB 다.
  - 이어 돌 때 체크포인트의 스위치·시드가 다르면 멈춘다.

## 4. 필요한 입력(서버 경로)

| 입력 | 경로 | 만든 곳 |
|---|---|---|
| 원천 거래 | `/workspace/IBM_AML_dataset` | IBM AML HI-Large |
| DuckDB(피처 단계까지) | `/tmp/tree9_work/db_full` | 태훈님 `daily_pipeline.py`(`/tmp/tree9_work/code/CODE/`, 코드 지문 검사) |
| 하루 캐시 85일(받은 문맥 · 대상 · 라벨) | `/tmp/gnn36_full` | 노트북 A(또는 §4 `RUN_CACHE = True`, 약 4시간) |
| 나가는 문맥 캐시 85일 | `/tmp/gnn36_out` | 이 노트북 `FIX_RUN_OUT_CACHE=1` |
| `amt_z_pay_ccy` | `common/amt_z_pay_ccy_f32.npy` | 노트북 A §2 |
| 라벨(9클래스 · 패턴 밖 표시) | `/tmp/tree9_work/labels_raw.npz` | `tree9_lib.label_arrays` |
| CSSMC r300 정상 선택 | `HI-Large_EDA/us_cmp_20260922/indices/cssmc_r300.npy`(sha1 `c237ac91…`) | 09-22 언더샘플링 비교 |
| 원본 체크포인트(§10 대조용) | `models/gnn36_cssmc_r300/ep3.pt` | 노트북 B |

- `/tmp` 는 컨테이너가 다시 만들어지면 사라진다. 캐시가 없으면 먼저 만들어야 한다.

---
아래 "상세" 머리말부터는 수정판 노트북의 설명과 코드를 그대로 옮긴 것이다.

## 상세 — 수정판 머리말(고칠 후보 · 스위치 · 09-29 결정)

손은총 · 2026-09-29 · 상태 **제안·미결** · 사용자 지시: "원래 있던 원본의 복사본을 하나 만들어서 거기서 수정. 고칠 후보별 준비 조건에 따라서. ipynb 형태로"

- **원본**: `TGNN9클래스_전체코드_한파일_20260929.ipynb`(노트북 B 와 같은 계산임을 자체 검사로 확인한 파일). 원본은 건드리지 않았다.
- **복사본(이 파일)**: 생성기 `_builders/build_nb_fix.py` 가 원본을 복사한 뒤 아래를 바꿨다.
  - §1~§7 의 원본 코드는 그대로다.
  - 새로 넣은 곳: `§1-수정`, `§7-수정`(후보 5개 · 수정 모델), §8·§9·§10(학습 · 채점 · 자체 검사).
- **원칙**
  - 스위치를 **모두 끄면 원본과 같은 계산**이다. §10 에서 학습된 epoch 3 가중치로 점수 차이 0 을 확인한다.
  - 팔(arm) 하나에 후보 **하나만** 켠다(한 번에 하나씩). **예외(09-29): `_m` 팔은 병합+셔플 위에 쌓는다 — 바로 앞 `_m` 팔과만 비교한다**(revmp_ego_m − revmp_m = ego 효과).

## 고칠 후보와 준비 조건

| 준비 조건 | 후보 | 팔 이름 | 무엇을 바꾸나 | 판단 필요 |
|---|---|---|---|---|
| **[A] 캐시 그대로 · 코드만** | 1 입력 표준화 + 층 정규화 | `norm`(둘 다) · `inorm` · `lnorm` | 연속 열 25개를 학습 날 통계로 z-정규화(이진 11열은 그대로), 각 TransformerConv 뒤 LayerNorm | — |
| | 2 배치 손실 균형 | `lossbal` | 배치 평균(배치 안 Σw 로 나눔) → epoch 전체 기준 고정 분모 W̄ 로 나눔. 행마다 손실 몫이 w/W̄ 로 일정 | — |
| | 3 규칙적 섞기 | `interleave` | 하루 안 학습 순서를 `i×k mod n`(k = 4099 부터 n 과 서로소)으로 재배열. 난수 없음. 그래프·날짜 순서는 그대로 | **무작위 금지 규칙과 맞는지** |
| | 4 dropout · weight decay | `reg` | 어텐션 dropout 0.2 · 헤드 입력 dropout 0.1 · AdamW weight decay 1e-5(FraudGT 설정값, 조사 에이전트 추출 · 미재확인) | **dropout 은 난수 사용(시드 고정)** |
| **[B] 캐시 추가 필요** | 5 역방향 메시지(+ ego) | `revmp` · `revmp_ego` | 보낸 계좌마다 **보낸 거래 최근 10건** 문맥을 새로 만들어(`/tmp/gnn36_out`, 원본 캐시는 그대로) 양방향 2홉. 층마다 받은 쪽 conv + 보낸 쪽 conv(자기 값 없이) 합산. ego = 분류할 거래의 양 끝 계좌 표시 1열 | ego 는 **입력 열 1개 추가**(36피처 결정과 충돌하는지) |

- **포트 번호는 넣지 않았다.** 문헌(Multi-GNN)은 전체 이력에서 이웃에 시간순 번호를 붙인다. 우리 문맥은 계좌당 10건이라 정의가 달라져 그대로 옮길 수 없다. 필요하면 따로 설계한다.
- **결과에 영향 없는 부수 변경**
  - 체크포인트에 메타데이터(피처·클래스 순서, cfg, 스위치, CSSMC sha1, 정규화 통계) 저장
  - CSSMC 파일 지문 검사
  - 패턴 밖 세탁 확률(`Po`) 저장
  - 조건부 8종 정답률(정답 패턴 행에서 8개 유형 중 argmax) 지표 추가

## 09-29 사용자 결정 반영(팀장 피드백 `message.txt` 뒤)
- **reg 제외**: 진단 G 에서 train 유형 정답률조차 0.11~0.13(과소적합)이라 과적합 대책은 뺀다.
- **라벨 병합 = 패턴 밖 세탁을 넣는다**(`label_merge`): 패턴 밖 세탁을 정상 클래스(8, API 의 0 NORMAL)로 학습·평가에 넣는다. 08-30 팀 결정으로 돌아가고, 09-22 의 "제거" 결정을 되돌린다.
  - 학습: CSSMC 는 진짜 정상만 골랐다. 그래서 train 패턴 밖 세탁 **43,095행을 전부** 정상 클래스(가중치 1)로 더한다. 학습 행은 16,806,937 → 16,850,032 이고, 가중치 합의 0.13% 다.
  - 평가: val 33,633,093 → 33,649,784행, test 30,390,801 → 30,407,864행(09-29 캐시에서 셈). 병합에서 `세탁여부_AP` 는 "패턴 세탁 여부"다(패턴 밖은 음성).
- **시드 셔플**(`shuffle`): 하루 안 학습 행 순서를 `np.random.default_rng(시드 + epoch(0부터) + 날짜의 dayofyear).permutation` 으로 섞는다.
  - 태훈님 원본 `daily_pipeline.py` 545줄과 같은 시드 식이다. 원본은 그날 거래 전체를 섞고, 여기서는 학습 행만 섞는다.
  - 같은 시드면 매번 같은 순서라 재현된다. 무작위 금지 규칙의 예외로 사용자가 승인했다(09-29).
- **ego 포함**: `revmp_ego_m`. 팀장 설명("판정할 거래 양끝 계좌에 flag")과 달리 **배치에 든 모든 분류 대상 거래(4,096건)의 양 끝 계좌**에 1 을 붙인다(배치 단위, Multi-GNN 구현도 배치 단위로 기억하나 원문 코드 미확인). 그래서 점수가 배치 구성에 따라 달라진다.
- **`_m` 팔은 채점(val·test)도 시드 셔플한 배치로 한다**(`split_scores`, 순서 시드 `[SEED, 7919, dayofyear]`, 점수는 원래 행 순서로 되돌림).
  - ego 와 양방향 샘플러(revmp — 받은 목록·보낸 목록이 따로 잘림)는 점수가 배치 구성에 따라 달라진다.
  - 학습은 섞인 배치, 채점은 시간순 배치면 두 분포가 어긋난다(검증 08:1x: 다른 대상의 표시 계좌를 보는 비율 섞은 배치 39% vs 시간순 42~60%).
  - 받은 쪽만 보는 샘플러(shuffle_m)는 점수가 배치와 무관해 결과가 같다.
- **interleave · lossbal 도 연쇄에서 뺐다(사용자 결정 아님 — 이 판단은 보고함)**
  - interleave 는 셔플과 같은 문제를 겨누고 같이 켤 수 없다.
  - lossbal 은 셔플 뒤 유형별 실효 가중치가 8종 모두 0.98~1.02 가 되어(셔플 전 0.27~1.77, 검증 08:1x 재계산) 겨눌 왜곡이 남지 않는다.
- **클래스 8 의 뜻**: 병합 팔에서 열 이름 `정상`·`AP_정상`·`세탁여부_AP` 는 각각 "정상 + 패턴 밖 세탁(= 패턴아님)", "패턴 세탁 여부"다. 기록에 `라벨` 열(병합/제거)을 둔다.
  - 병합 팔 채점에는 패턴 밖 행을 뺀 macro-AP·바닥대비(`*_패턴밖제외`)도 남겨 앞 팔과 같은 행으로 비교한다.
  - 조건부 8종은 병합과 무관하다. 바닥대비는 거의 무관하다(norm 에서 0.0004 차).
- **반영하지 않은 팀장 피드백**
  - 클래스 순서 API 맞추기: 내보낼 때 `P_api = P[:, [8,0,1,4,3,2,7,5,6]]`(G-SCATTER = GATHER-SCATTER, S-GATHER = SCATTER-GATHER 로 봄 — 팀 확인 필요).
  - 이웃 상한 10 → 64~100: 날짜별 캐시로는 수백 GB 라 보류했다. 나가는 문맥 캐시도 10건이라 상한을 올리면 둘 다 다시 만든다.
  - 3시드: 최종 설정만.
  - revmp·ego 는 태훈님 라이브러리 수정을 기다리지 않고 먼저 넣었다(사용자 결정).
- **시드 주의**: 셔플 시드 식이 정수 합(SEED + epoch + dayofyear)이라 시드 0 의 epoch e+1 과 시드 1 의 epoch e 는 같은 순서다. 3시드는 15 이상 떨어뜨린다(예: 42 · 0 · 100). `run_fix_arms.sh` 는 출력 노트북·로그 이름에 시드를 안 붙이므로 시드 반복은 별도 연쇄로 돌린다.
- **`_m` 팔은 쌓아 올린다**(병합 + 셔플 위에 하나씩): `shuffle_m` → `revmp_m` → `revmp_ego_m`.
  - 앞의 팔들(off·norm·…)은 병합 전 라벨이라 macro-AP 를 직접 비교하지 않는다.
  - 대신 병합과 무관한 조건부 8종 정답률과 자기 바닥 대비(`바닥대비`)로 비교한다.
- **시드**: `FIX_SEED`(기본 42). 42 가 아니면 출력 폴더에 `_s<시드>` 가 붙는다. 최종 설정만 3시드(42·0·1)로 다시 돈다.
- **새 기록 열**(`학습기록.csv`)
  - `val_세탁여부_ap`, `val_조건부8종`
  - `val_바닥` = (정상 AP + 세탁 여부 AP) ÷ 9. 유형 정보가 전혀 없을 때의 macro-AP 다.
  - `val_바닥대비` = val macro-AP − `val_바닥`. 0 보다 커야 유형 머리가 쓸모 있다.
  - `train_유형CE`·`train_조건부8종`: 학습 중 세탁 행의 조건부 8종 CE·정답률. 사전분포 엔트로피 2.03 아래로 내려가야 유형을 배우는 것이다.

## 실행 스위치 (환경 변수로 켠다 — 기본은 가벼운 자체 검사만)
| 변수 | 기본 | 뜻 |
|---|---|---|
| `FIX_ARM` | `off` | 팔 이름(위 표) |
| `FIX_RUN_SELFCHECK` | 1 | §10 자체 검사(CPU 몇 분, 캐시를 mmap 으로 읽어 메모리 약 1~2 GiB) |
| `FIX_RUN_OUT_CACHE` | 0 | [B] 나가는 문맥 캐시 85일(DuckDB 사용 — **반드시 단독 실행**, 시간 미측정 · 약 1~2시간 추정) |
| `FIX_RUN_TRAIN` · `FIX_RUN_EVAL` | 0 | GPU 학습 · 채점(팔마다 약 1~3시간, revmp 는 더 김) |
| `FIX_SEED` | 42 | 모델 초기화 · 시드 셔플의 시드. 42 가 아니면 출력 폴더 `_fix/<팔>_s<시드>` |
| `FIX_SMOKE` | 0 | 1 이면 train 2일 · val 1일 · test 1일 · 1 epoch(코드 확인용) |

실행 예(LightGBM 등 다른 무거운 작업이 없을 때):
```bash
cd /workspace/Final_preprocess/HI-Large_EDA/tgnn36_20260928
FIX_ARM=norm FIX_SMOKE=1 FIX_RUN_SELFCHECK=0 FIX_RUN_TRAIN=1 FIX_RUN_EVAL=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 TGNN9클래스_수정판_한파일_20260929.ipynb --output-dir _fix/norm --output smoke.ipynb
```
여러 팔을 차례로 돌릴 때는 `run_fix_arms.sh`(LightGBM 연쇄가 끝날 때까지 기다렸다가 팔마다 스모크 → 본 학습, 실패하면 멈춤)를 쓴다. **사용자 승인 전에는 띄우지 않는다.**


In [ ]:
import os
RUN_SELFCHECK = os.environ.get('FIX_RUN_SELFCHECK', '1') == '1'
RUN_OUT_CACHE = os.environ.get('FIX_RUN_OUT_CACHE', '0') == '1'    # [B] 나가는 문맥 캐시 — 반드시 단독
RUN_TRAIN = os.environ.get('FIX_RUN_TRAIN', '0') == '1'
RUN_EVAL = os.environ.get('FIX_RUN_EVAL', '0') == '1'
SMOKE = os.environ.get('FIX_SMOKE', '0') == '1'
RUN_CACHE = False                                                 # 원본 §4 의 캐시 재생성(끔 — 원본 캐시를 읽기만 한다)
# 팔 = 원본에서 후보 하나만 바꾼 설정
ARMS = {
    'off':        dict(),                                                        # 원본과 같은 계산
    'norm':       dict(input_norm=True, layer_norm=True),                        # 후보 1
    'inorm':      dict(input_norm=True),                                         # 후보 1 을 나눠 볼 때
    'lnorm':      dict(layer_norm=True),
    'lossbal':    dict(loss_balance=True),                                       # 후보 2
    'interleave': dict(interleave=True),                                         # 후보 3 (판단 필요)
    'reg':        dict(dropout_attn=0.2, dropout_head=0.1, weight_decay=1e-5),   # 후보 4 (판단 필요)
    'revmp':      dict(reverse_mp=True),                                         # 후보 5 — [B] 캐시 필요
    'revmp_ego':  dict(reverse_mp=True, ego=True),                               # 후보 5 + ego (판단 필요)
    # 09-29 사용자 결정: 라벨 병합(패턴 밖 세탁 → 정상 클래스) + 시드 셔플 위에 쌓는다
    'shuffle_m':   dict(label_merge=True, shuffle=True),
    'revmp_m':     dict(label_merge=True, shuffle=True, reverse_mp=True),                 # [B] 캐시 필요
    'revmp_ego_m': dict(label_merge=True, shuffle=True, reverse_mp=True, ego=True),       # [B] 캐시 필요
}
DEFAULTS = dict(input_norm=False, layer_norm=False, loss_balance=False, interleave=False,
                dropout_attn=0.0, dropout_head=0.0, weight_decay=0.0, reverse_mp=False, ego=False,
                label_merge=False, shuffle=False)
FIX_ARM = os.environ.get('FIX_ARM', 'shuffle_m')                  # 통합본: 기본 팔 = 병합 + 시드 셔플
assert FIX_ARM in ARMS, f'모르는 팔 {FIX_ARM} — 가능한 팔: {list(ARMS)}'
FIX = dict(DEFAULTS, **ARMS[FIX_ARM])
assert not (FIX['shuffle'] and FIX['interleave']), '시드 셔플과 규칙적 섞기는 같이 켜지 않는다'
SEED = int(os.environ.get('FIX_SEED', '42'))                      # 모델 초기화 · 시드 셔플(cfg.seed 는 DB 설정이라 건드리지 않는다)
print('팔', FIX_ARM, '· 시드', SEED, '· 스모크' if SMOKE else '', FIX)
print(dict(RUN_SELFCHECK=RUN_SELFCHECK, RUN_OUT_CACHE=RUN_OUT_CACHE, RUN_TRAIN=RUN_TRAIN, RUN_EVAL=RUN_EVAL))

## §1 설정값 (태훈님 `Config` 그대로 + plan.json data_config)

In [ ]:
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '3')
import sys, json, time, gc, math, hashlib, inspect
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np, pandas as pd
for _p in ('/workspace/.pylibs/duckdb', '/workspace/Final_preprocess/HI-Large_EDA/_deps'):
    if _p not in sys.path: sys.path.insert(0, _p)
import duckdb, torch
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense.linear import Linear
from torch_geometric.utils import softmax as pyg_softmax

CODE_FILE = Path('/tmp/tree9_work/code/CODE/daily_pipeline.py')          # DuckDB 를 만든 태훈님 원본(코드 지문 검사용)
PLAN_SPLIT = json.loads((CODE_FILE.parents[1] / 'experiments/split_60_20_20/plan.json').read_text())

@dataclass(frozen=True)
class Config:                                            # daily_pipeline.py 41-62줄 그대로
    source: str = '/workspace/IBM'
    output: str = 'daily_v1'
    calibration_start: str = '2022-08-01'
    train_start: str = '2022-08-08'
    val_start: str = '2022-09-01'
    test_start: str = '2022-10-01'
    test_end: str = '2022-11-01'
    history_days: int = 30
    incoming_neighbors: int = 10
    target_batch_size: int = 4096
    duplicate_policy: str = 'keep'
    memory_limit: str = '3GB'
    threads: int = 2
    epochs: int = 5
    patience: int = 2
    hidden: int = 128
    heads: int = 2
    lr: float = 0.002
    pos_weight: float = 300.0
    seed: int = 42

_d = dict(PLAN_SPLIT['data_config']); _d.update(source='/workspace/IBM_AML_dataset', output='/tmp/tree9_work/db_full', threads=4, memory_limit='14GB')
cfg = Config(**_d)                                       # tree9_lib.make_cfg 와 같은 값(09-24·09-28 DB 를 만든 설정)
EDA = Path('/workspace/Final_preprocess/HI-Large_EDA'); W = EDA / 'tgnn36_20260928'
OUT = W / '_onefile'; (OUT / 'results').mkdir(parents=True, exist_ok=True); (OUT / 'models').mkdir(exist_ok=True)
FC = Path('/tmp/gnn36_full')                             # 노트북 A 가 만든 캐시(읽기만)
SPLIT_DAYS = {'train': ('2022-08-08', '2022-09-27'), 'val': ('2022-09-28', '2022-10-14'), 'test': ('2022-10-15', '2022-10-31')}   # tree9_lib.SPLIT_DAYS
DAYS = {s: [str(d.date()) for d in pd.date_range(*SPLIT_DAYS[s], freq='D')] for s in SPLIT_DAYS}
K = 9; TYPES = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER', 'BIPARTITE', 'STACK', 'RANDOM']; CLASS9 = TYPES + ['정상']
MAX_EP, PAT, RULES = 15, 3, {'규칙15_3': (15, 3), '규칙15_2': (15, 2), '규칙5_2': (5, 2)}   # 사용자 결정 09-28
POS_W = 300.0
print({k: v for k, v in asdict(cfg).items() if k in ('hidden', 'heads', 'lr', 'target_batch_size', 'history_days', 'incoming_neighbors', 'seed')})

## §1-수정 — 팔별 출력 폴더 · 스모크 기간 · 메모리 확인 도구

In [ ]:
RUN_NAME = 'out_cache' if RUN_OUT_CACHE else FIX_ARM + (f'_s{SEED}' if SEED != 42 else '')   # 시드 42 는 이름 그대로 · 캐시 만들기는 팔과 헷갈리지 않게
OUT = W / '_team' / (RUN_NAME + ('_smoke' if SMOKE else ''))           # 통합본: 팔별 폴더(서버에서 도는 _fix/ 결과를 덮지 않게)
(OUT / 'results').mkdir(parents=True, exist_ok=True); (OUT / 'models').mkdir(exist_ok=True)
COMMON_FIX = W / '_fix' / 'common'; COMMON_FIX.mkdir(parents=True, exist_ok=True)
FC_OUT = Path('/tmp/gnn36_out')                                  # [B] 나가는 문맥(원본 캐시 /tmp/gnn36_full 는 읽기만)
SC = Path('/tmp/gnn36_scores') / ('team_' + RUN_NAME + ('_smoke' if SMOKE else '')); SC.mkdir(parents=True, exist_ok=True)
CSSMC_PATH = EDA / 'us_cmp_20260922/indices/cssmc_r300.npy'; CSSMC_SHA1 = 'c237ac915b91069064eb1c29183ad2850dd31823'
if SMOKE:
    DAYS = {'train': DAYS['train'][:2], 'val': DAYS['val'][:1], 'test': DAYS['test'][:1]}
    MAX_EP, PAT = 1, 1

def anon_gib():
    for l in open('/sys/fs/cgroup/memory.stat'):
        if l.startswith('anon '): return int(l.split()[1]) / 2**30
    return float('nan')

def heavy_jobs():
    """지금 도는 무거운 작업(LightGBM 연쇄 · 다른 nbconvert 학습)."""
    import subprocess
    ps = subprocess.run(['ps', '-eo', 'pid,stat,args'], capture_output=True, text=True).stdout.splitlines()[1:]
    me, out = str(os.getpid()), []
    for l in ps:
        parts = l.split(None, 2)
        if len(parts) < 3 or parts[0] == me or 'Z' in parts[1]:
            continue                                                  # 좀비(<defunct>)는 자원을 쓰지 않으므로 무시(09-29 C 중단 뒤 남은 좀비)
        if any(k in parts[2] for k in ('run_chain_BC.sh', 'C_LightGBM9클래스', 'A_준비_DuckDB')):
            out.append(l.strip()[:120])
    return out

def status_fix(t):
    line = f'[{time.strftime("%F %T")} UTC] [FIX:{RUN_NAME}{"/smoke" if SMOKE else ""}] {t}'
    (OUT / 'STATUS.txt').write_text(line + '\n')
    with open(W / 'STATUS_log.txt', 'a') as f: f.write(line + '\n')
    print(line, flush=True)

def _sha1(p):
    h = hashlib.sha1()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): h.update(b)
    return h.hexdigest()

NB_SHA1 = _sha1(W / 'TGNN9클래스_통합본_팀장전달_20260929.ipynb')      # 실행 시작 때 노트북 지문(어느 코드로 돌았는지 meta 에 남김)
print('출력', OUT, '· anon', round(anon_gib(), 2), 'GiB · 무거운 작업', heavy_jobs() or '없음')

## §2 36피처 이름 (노드 19 · 엣지 17) — 태훈님 코드의 원래 23·18열에서 고르는 위치

In [ ]:
EDGE_FEATURES = ['pair_prior_count','pair_gap_minutes','pair_count_1h','sender_gap_minutes',
 'sender_count_1h','receiver_gap_minutes','receiver_count_1h','paid_currency_z',
 'pair_history_count','reverse_history_count','pair_age_at_day_start',
 'same_entity','same_known_country','both_countries_known','same_currency','is_self_transfer',
 'paid_currency_unseen','received_currency_unseen']                  # daily_pipeline.py 28-32줄
ENTITY_TYPES = ['Corporation','Partnership','Sole Proprietorship','Individual','Country','Direct']
NODE_FEATURES = ['log_out_count','log_in_count','log_out_neighbors','log_in_neighbors',
 'log_self_count','log_reciprocal_neighbors','net_count_ratio','reciprocity_ratio',
 'mean_paid_currency_z','mean_received_currency_z','paid_outlier_fraction',
 'received_outlier_fraction','log_currency_count','log_active_days'] + [
 'type_'+x for x in ENTITY_TYPES] + ['log_entity_observed_accounts','log_entity_out_diversity','log_entity_in_diversity']   # 33-37줄
SCH = pd.read_csv('/workspace/Final_preprocess/aml_feature_schema.csv', encoding='utf-8-sig')
NODE19 = list(SCH.query("tensor=='node'").sort_values('col_index')['feature'])
EDGE17 = list(SCH.query("tensor=='edge'").sort_values('col_index')['feature'])
NSEL = np.array([NODE_FEATURES.index(f) for f in NODE19]); ESEL = np.array([EDGE_FEATURES.index(f) for f in EDGE17[:16]])
assert len(NODE19) == 19 and len(EDGE17) == 17 and EDGE17[-1] == 'amt_z_pay_ccy' and list(ESEL) == list(range(16))
print('빼는 노드', [f for f in NODE_FEATURES if f not in NODE19], '· 빼는 엣지', EDGE_FEATURES[16:], '· 더하는 엣지 amt_z_pay_ccy')

## §3 하루 그래프 — `load_day` (daily_pipeline.py 343-403줄 그대로 + 문맥 거래 번호 한 줄)
- 노드 = 30일 창 `[D−29, D]` 활동 계좌, 노드 피처 23개(여기서 19개를 §4 에서 고름)
- 문맥 엣지 = 계좌마다 **받은 거래 최근 10건**(`QUALIFY row_number() OVER(PARTITION BY dst ORDER BY ts DESC, edge_id DESC) <= 10`)
- 대상 = D일 거래 전부

In [ ]:
def sql_string(x): return "'"+str(x).replace("'","''")+"'"
def file_fingerprint(p):
    p=Path(p); s=p.stat()
    with p.open('rb') as f:
        head=f.read(1024*1024); f.seek(max(0,s.st_size-1024*1024)); tail=f.read()
    return {'path':str(p.resolve()),'bytes':s.st_size,'mtime_ns':s.st_mtime_ns,'head_tail_sha256':hashlib.sha256(head+tail).hexdigest()}
def base(cfg):
    p=Path(cfg.output);p.mkdir(parents=True,exist_ok=True);return p
def connect(cfg,read_only=False):                        # 84-88줄
    p=base(cfg);c=duckdb.connect(str(p/'daily.duckdb'),read_only=read_only)
    c.execute(f'SET memory_limit={sql_string(cfg.memory_limit)}');c.execute(f'SET threads={cfg.threads}')
    c.execute(f"SET temp_directory={sql_string(p/'spill')}")
    c.execute('SET preserve_insertion_order=true');return c
def require_stage(cfg,stage):                            # 94-101줄 (코드 지문만 CODE_FILE 로 — DB 를 만든 원본)
    p=base(cfg)/(stage+'.json');assert p.exists(),f'Run {stage} first'
    m=json.loads(p.read_text());assert m.get('complete'),f'Incomplete {stage}'
    assert m['config']==asdict(cfg),'Configuration changed: use a new output directory or rebuild stages'
    assert m['code_sha256']==hashlib.sha256(CODE_FILE.read_bytes()).hexdigest(),'Code changed: rebuild into a new output directory'
    if 'sources' in m:
        for spec in m['sources'].values():assert file_fingerprint(spec['path'])==spec,'Source changed; use new output directory'
    return m
def finite(name,x):
    x=np.asarray(x);bad=~np.isfinite(x)
    if bad.any():raise ValueError(f'{name}: {int(bad.sum())} NaN/Inf values')
    return x

@dataclass
class DayData:                                           # 328-340줄
    day: str
    node_ids: np.ndarray
    node_x: np.ndarray
    context_src: np.ndarray
    context_dst: np.ndarray
    context_x: np.ndarray
    target_ids: np.ndarray
    target_src: np.ndarray
    target_dst: np.ndarray
    target_x: np.ndarray
    labels: np.ndarray | None

_CTX_IDS = [None]                                        # 문맥 거래 번호(원본에 없던 한 줄이 여기에 넣는다)
def load_day(cfg,day,with_labels=False):                 # 343-403줄
    require_stage(cfg,'features');day=pd.Timestamp(day).normalize();start=day-pd.Timedelta(days=cfg.history_days-1)
    if day<pd.Timestamp(cfg.train_start):raise ValueError('No predictions during baseline calibration')
    c=connect(cfg,read_only=True)
    try:
        c.execute(f"""CREATE TEMP VIEW hist AS SELECT f.*,t.payment_currency,t.receiving_currency,
           ks.entity_id AS source_entity,kd.entity_id AS target_entity
           FROM features f JOIN tx t USING(edge_id) JOIN kyc ks ON ks.node_id=f.src JOIN kyc kd ON kd.node_id=f.dst
           WHERE f.day>={sql_string(str(start.date()))}::DATE AND f.day<={sql_string(str(day.date()))}::DATE""")
        agg=c.execute("""WITH outgoing AS (SELECT src AS node_id,count(*) AS out_n,count(DISTINCT dst) AS out_u,
              count(*) FILTER(WHERE src=dst) AS self_n,avg(paid_currency_z) AS out_z,
              avg(CASE WHEN abs(paid_currency_z)>=3 THEN 1.0 ELSE 0 END) AS out_tail FROM hist GROUP BY src),
          incoming AS (SELECT dst AS node_id,count(*) AS in_n,count(DISTINCT src) AS in_u,avg(received_currency_z) AS in_z,
              avg(CASE WHEN abs(received_currency_z)>=3 THEN 1.0 ELSE 0 END) AS in_tail FROM hist GROUP BY dst),
          pairs AS (SELECT DISTINCT src,dst FROM hist WHERE src!=dst),
          reciprocal AS (SELECT a.src AS node_id,count(*) AS reciprocal_n FROM pairs a JOIN pairs b ON a.src=b.dst AND a.dst=b.src GROUP BY a.src),
          currency_set AS (SELECT src AS node_id,payment_currency AS currency FROM hist UNION SELECT dst,receiving_currency FROM hist),
          currencies AS (SELECT node_id,count(*) AS n_currency FROM currency_set GROUP BY node_id),
          active AS (SELECT src AS node_id,day FROM hist UNION SELECT dst,day FROM hist),
          active_days AS (SELECT node_id,count(*) AS n_days FROM active GROUP BY node_id),
          ent_pairs AS (SELECT DISTINCT source_entity,target_entity FROM hist WHERE source_entity IS NOT NULL AND target_entity IS NOT NULL),
          ent_out AS (SELECT source_entity AS entity_id,count(*) AS ent_out_n FROM ent_pairs GROUP BY source_entity),
          ent_in AS (SELECT target_entity AS entity_id,count(*) AS ent_in_n FROM ent_pairs GROUP BY target_entity)
          SELECT k.*,CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER(PARTITION BY k.entity_id) END AS observed_entity_accounts,coalesce(o.out_n,0) AS out_n,coalesce(i.in_n,0) AS in_n,coalesce(o.out_u,0) AS out_u,
           coalesce(i.in_u,0) AS in_u,coalesce(o.self_n,0) AS self_n,coalesce(r.reciprocal_n,0) AS reciprocal_n,
           coalesce(o.out_z,0) AS out_z,coalesce(i.in_z,0) AS in_z,coalesce(o.out_tail,0) AS out_tail,
           coalesce(i.in_tail,0) AS in_tail,cu.n_currency,ad.n_days,coalesce(eo.ent_out_n,0) AS ent_out_n,
           coalesce(ei.ent_in_n,0) AS ent_in_n
          FROM active_days ad JOIN kyc k USING(node_id) LEFT JOIN outgoing o USING(node_id)
          LEFT JOIN incoming i USING(node_id) LEFT JOIN reciprocal r USING(node_id)
          JOIN currencies cu USING(node_id) LEFT JOIN ent_out eo ON k.entity_id=eo.entity_id
          LEFT JOIN ent_in ei ON k.entity_id=ei.entity_id ORDER BY node_id""").fetchdf()
        if agg.empty:raise ValueError(f'No graph history on {day.date()}')
        v=lambda name:agg[name].to_numpy(np.float64)
        outn,inn=v('out_n'),v('in_n');ou,iu=v('out_u'),v('in_u');rec=v('reciprocal_n')
        fields=[np.log1p(outn),np.log1p(inn),np.log1p(ou),np.log1p(iu),np.log1p(v('self_n')),np.log1p(rec),
          (outn-inn)/np.maximum(outn+inn,1),rec/np.maximum(ou+iu,1),v('out_z'),v('in_z'),v('out_tail'),v('in_tail'),
          np.log1p(v('n_currency')),np.log1p(v('n_days'))]
        fields += [(agg.entity_type==typ).to_numpy(np.float64) for typ in ENTITY_TYPES]
        fields += [np.log1p(v('observed_entity_accounts')),np.log1p(v('ent_out_n')),np.log1p(v('ent_in_n'))]
        nx=finite('daily node features',np.stack(fields,axis=1).astype(np.float32))
        node_ids=agg.node_id.to_numpy(np.int64)
        names=','.join(EDGE_FEATURES)
        context=c.execute(f"""SELECT edge_id,src,dst,{names} FROM hist
          QUALIFY row_number() OVER(PARTITION BY dst ORDER BY ts DESC,edge_id DESC)<={cfg.incoming_neighbors}
          ORDER BY dst,ts DESC,edge_id DESC""").fetchdf()
        targets=c.execute(f"""SELECT f.edge_id,f.src,f.dst,{','.join('f.'+x for x in EDGE_FEATURES)}
          {',t.y' if with_labels else ''} FROM features f {'JOIN tx t USING(edge_id)' if with_labels else ''}
          WHERE f.day={sql_string(str(day.date()))}::DATE ORDER BY f.edge_id""").fetchdf()
        if targets.empty:raise ValueError(f'No target transactions on {day.date()}')
        _CTX_IDS[0]=context.edge_id.to_numpy(np.int64)   # ← 원본에 없는 한 줄(노트북 A 와 같음)
        return DayData(str(day.date()),node_ids,nx,context.src.to_numpy(np.int64),context.dst.to_numpy(np.int64),
          finite('context edges',context[EDGE_FEATURES].to_numpy(np.float32)),targets.edge_id.to_numpy(np.int64),
          targets.src.to_numpy(np.int64),targets.dst.to_numpy(np.int64),finite('target edges',targets[EDGE_FEATURES].to_numpy(np.float32)),
          targets.y.to_numpy(np.int8) if with_labels else None)
    finally:c.close()

## §4 36피처로 고르기 + `amt_z_pay_ccy` 붙이기 · 9클래스 라벨 (노트북 A §4 · `tree9_lib.label_arrays`)

In [ ]:
AZ = np.load(W / 'common/amt_z_pay_ccy_f32.npy', mmap_mode='r')          # 노트북 A §2 에서 계산(태훈님 코드와 미대조)
INTERIM = Path('/workspace/processed_9class/_nb_scratch/interim/HI-Large'); MAIN = Path('/workspace/processed_9class/HI-Large_main')
def label_arrays():                                      # tree9_lib.py 183-197줄 — 원본 행 번호 → 내 y9 · is_pos(모집단 밖 −1)
    f = Path('/tmp/tree9_work/labels_raw.npz')
    if f.exists():
        z = np.load(f); return z['y9'], z['is_pos']
    n_raw = int(json.loads((INTERIM / 'manifest.json').read_text())['n_rows'])
    p2r = np.load(EDA / 'reprocess/align/new_proc2raw.npy'); idx = np.load(MAIN / 'index_full.npz')
    y9 = np.full(n_raw, -1, np.int8); y9[p2r] = idx['y9']; ip = np.full(n_raw, -1, np.int8); ip[p2r] = idx['is_pos'].astype(np.int8)
    return y9, ip
Y9_RAW, IP_RAW = label_arrays()
def y9_of(eid):
    """0~7 패턴 · 8 정상 · −1 = 뺄 행(패턴 밖 세탁, 내 모집단 밖). 사용자 결정 09-22."""
    y9, ip = Y9_RAW[eid], IP_RAW[eid]
    return np.where(y9 < 0, -1, np.where(y9 < 8, y9, np.where(ip == 1, -1, 8))).astype(np.int8)
def to36(day):
    """load_day 결과(노드 23 · 엣지 18) → 노드 19 · 엣지 16 + amt_z = 17."""
    ctx = _CTX_IDS[0]; az_c, az_t = np.asarray(AZ[ctx]), np.asarray(AZ[day.target_ids])
    assert np.isfinite(az_c).all() and np.isfinite(az_t).all()
    return DayData(day.day, day.node_ids, np.ascontiguousarray(day.node_x[:, NSEL]), day.context_src, day.context_dst,
                   np.concatenate([day.context_x[:, ESEL], az_c[:, None]], 1).astype(np.float32), day.target_ids, day.target_src, day.target_dst,
                   np.concatenate([day.target_x[:, ESEL], az_t[:, None]], 1).astype(np.float32), day.labels), ctx

CACHE_FIELDS = ['node_ids', 'node_x', 'context_src', 'context_dst', 'context_x', 'target_ids', 'target_src', 'target_dst', 'target_x', 'labels']
def load_cached(d, root=FC):
    dd = root / f'day={d}'
    return DayData(day=d, **{f: np.load(dd / f'{f}.npy') for f in CACHE_FIELDS}), np.load(dd / 'y9.npy')

if RUN_CACHE:                                            # 약 4시간 — 기존 캐시를 덮지 않도록 다른 폴더에
    FC_NEW = Path('/tmp/gnn36_onefile_cache'); FC_NEW.mkdir(parents=True, exist_ok=True)
    for d in DAYS['train'] + DAYS['val'] + DAYS['test']:
        dd = FC_NEW / f'day={d}'
        if (dd / '_DONE').exists(): continue
        dd.mkdir(exist_ok=True); day36, ctx = to36(load_day(cfg, d, with_labels=True)); y = y9_of(day36.target_ids)
        for f in CACHE_FIELDS: np.save(dd / f'{f}.npy', getattr(day36, f))
        np.save(dd / 'context_ids.npy', ctx); np.save(dd / 'y9.npy', y); (dd / '_DONE').write_text(d); print(d, flush=True)
    FC = FC_NEW

## §5 이웃 샘플러 · 텐서 (daily_pipeline.py 406-434 · 462-464줄 그대로) — 받은 쪽 2홉

In [ ]:
class FixedNeighborSampler:
    """Deterministic two-hop incoming neighborhoods, independent of batch members."""
    def __init__(self,day):
        self.day=day
        self.dest,self.start,self.count=np.unique(day.context_dst,return_index=True,return_counts=True)
        assert np.all(day.context_dst[:-1]<=day.context_dst[1:])
    def incoming(self,nodes):
        nodes=np.unique(nodes);i=np.searchsorted(self.dest,nodes);ok=i<len(self.dest)
        ok[ok]=self.dest[i[ok]]==nodes[ok];i=i[ok]
        if not len(i):return np.empty(0,np.int64)
        counts=self.count[i];starts=self.start[i]
        offsets=np.arange(counts.sum())-np.repeat(np.cumsum(counts)-counts,counts)
        return np.repeat(starts,counts)+offsets
    def batch(self,positions):
        d=self.day;positions=np.asarray(positions,dtype=np.int64)
        seeds=np.unique(np.r_[d.target_src[positions],d.target_dst[positions]])     # 대상 거래의 보낸·받은 계좌
        first=self.incoming(seeds);second=self.incoming(d.context_src[first])       # 1홉·2홉 = 들어오는 엣지만
        ci=np.union1d(first,second)
        nodes=np.unique(np.r_[seeds,d.context_src[ci],d.context_dst[ci]])
        ni=np.searchsorted(d.node_ids,nodes)
        assert (ni<len(d.node_ids)).all() and np.array_equal(d.node_ids[ni],nodes),'Unknown node mapping'
        ei=np.stack([np.searchsorted(nodes,d.context_src[ci]),np.searchsorted(nodes,d.context_dst[ci])])
        targets=np.stack([np.searchsorted(nodes,d.target_src[positions]),np.searchsorted(nodes,d.target_dst[positions])])
        return d.node_x[ni],ei,d.context_x[ci],targets,d.target_x[positions]

def tensor_batch(batch,device):
    return tuple(torch.as_tensor(x,dtype=torch.long if i in {1,3} else torch.float32,device=device) for i,x in enumerate(batch))

## §6 TransformerConv 층 (torch_geometric 2.8.0 `transformer_conv.py` 의 계산 부분)
층마다 받는 계좌 i 에 대해: `h_i = Σ_j softmax_j((W_q x_i)·(W_k x_j + W_e e_ji)/√C) · (W_v x_j + W_e e_ji) + W_skip x_i`
(j = i 로 돈을 보낸 이웃 계좌, e = 그 거래의 엣지 피처, 헤드별로 계산해 이어 붙임). 가중치 합이 1 이라 **가중 평균** — 이웃 수는 셀 수 없다.

In [ ]:
class TransformerConv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, concat=True, beta=False, dropout=0., edge_dim=None, bias=True, root_weight=True, **kwargs):
        kwargs.setdefault('aggr', 'add')                  # 합산(가중치가 softmax 라 결과는 가중 평균)
        super().__init__(node_dim=0, **kwargs)
        self.in_channels, self.out_channels, self.heads = in_channels, out_channels, heads
        self.beta = beta and root_weight; self.root_weight = root_weight; self.concat = concat
        self.dropout = dropout; self.edge_dim = edge_dim; self._alpha = None
        if isinstance(in_channels, int): in_channels = (in_channels, in_channels)
        self.lin_key = Linear(in_channels[0], heads * out_channels, bias=bias)
        self.lin_query = Linear(in_channels[1], heads * out_channels, bias=bias)
        self.lin_value = Linear(in_channels[0], heads * out_channels, bias=bias)
        if edge_dim is not None: self.lin_edge = Linear(edge_dim, heads * out_channels, bias=False)
        else: self.lin_edge = self.register_parameter('lin_edge', None)
        if concat:
            self.lin_skip = Linear(in_channels[1], heads * out_channels, bias=bias)
            self.lin_beta = Linear(3 * heads * out_channels, 1, bias=False) if self.beta else self.register_parameter('lin_beta', None)
        else:
            self.lin_skip = Linear(in_channels[1], out_channels, bias=bias)
            self.lin_beta = Linear(3 * out_channels, 1, bias=False) if self.beta else self.register_parameter('lin_beta', None)
        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin_key.reset_parameters(); self.lin_query.reset_parameters(); self.lin_value.reset_parameters()
        if self.edge_dim: self.lin_edge.reset_parameters()
        self.lin_skip.reset_parameters()
        if self.beta: self.lin_beta.reset_parameters()

    def forward(self, x, edge_index, edge_attr=None):
        H, C = self.heads, self.out_channels
        if isinstance(x, Tensor): x = (x, x)
        query = self.lin_query(x[1]).view(-1, H, C)       # 받는 계좌(i)
        key = self.lin_key(x[0]).view(-1, H, C)           # 보낸 이웃(j)
        value = self.lin_value(x[0]).view(-1, H, C)
        out = self.propagate(edge_index, query=query, key=key, value=value, edge_attr=edge_attr)
        self._alpha = None
        out = out.view(-1, H * C) if self.concat else out.mean(dim=1)
        if self.root_weight:
            x_r = self.lin_skip(x[1])                     # 자기 자신 값 더하기
            if self.lin_beta is not None:
                beta = self.lin_beta(torch.cat([out, x_r, out - x_r], dim=-1)).sigmoid(); out = beta * x_r + (1 - beta) * out
            else:
                out = out + x_r
        return out

    def message(self, query_i, key_j, value_j, edge_attr, index, ptr, size_i):
        if self.lin_edge is not None:
            assert edge_attr is not None
            edge_attr = self.lin_edge(edge_attr).view(-1, self.heads, self.out_channels)
            key_j = key_j + edge_attr                     # 키에 거래 피처
        alpha = (query_i * key_j).sum(dim=-1) / math.sqrt(self.out_channels)
        alpha = pyg_softmax(alpha, index, ptr, size_i)    # 받은 이웃끼리 합 1
        self._alpha = alpha
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        out = value_j
        if edge_attr is not None: out = out + edge_attr   # 값에도 거래 피처
        return out * alpha.view(-1, self.heads, 1)

## §7 모델 `DailyGNN9` (노트북 B §2 = daily_pipeline.py 448-459줄, 출력만 9개)

In [ ]:
class DailyGNN9(torch.nn.Module):
    def __init__(self, n_node=19, n_edge=17, k=K, conv=TransformerConv):
        super().__init__()
        self.conv1 = conv(n_node, cfg.hidden // cfg.heads, heads=cfg.heads, edge_dim=n_edge, dropout=0.)
        self.conv2 = conv(cfg.hidden, cfg.hidden // cfg.heads, heads=cfg.heads, edge_dim=n_edge, dropout=0.)
        self.head = torch.nn.Sequential(torch.nn.Linear(2 * cfg.hidden + n_edge, cfg.hidden), torch.nn.ReLU(), torch.nn.Linear(cfg.hidden, k))
    def forward(self, x, ei, ea, targets, target_attr):
        h = torch.relu(self.conv1(x, ei, ea)); h = torch.relu(self.conv2(h, ei, ea))
        return self.head(torch.cat([h[targets[0]], h[targets[1]], target_attr], dim=1))   # [보낸 | 받은 | 이 거래] → 9개 점수
print('파라미터', sum(p.numel() for p in DailyGNN9().parameters()))

## §7-수정 — 고칠 후보 5개 (준비 조건별)

아래 함수·클래스는 스위치(`FIX`)가 켜진 것만 동작한다. 모두 끄면 §8 학습은 원본 §8 과 같은 계산이다.

### [A] 캐시 그대로 · 코드만 바꾸는 것

#### 후보 1 — 입력 표준화(A1) + 층 정규화(A2)
- **표준화 통계**: **학습 날(08-08~09-27)만** 쓴다(누수 방지).
  - 노드 = 학습 날 캐시의 `node_x` 전 행(계좌 × 날)
  - 엣지 = 학습 날 `context_x` 전 행(모델이 메시지로 보는 거래 — 같은 거래가 여러 날 문맥에 나오면 여러 번 셈). 대상 거래(`target_x`)에도 같은 엣지 통계를 쓴다.
- **이진 열 11개**(스키마 `value_type=binary`)는 표준화하지 않는다. 표준편차가 0.004 인 열을 나누면 값이 250배로 튀기 때문이다.
- 통계는 `_fix/common/input_norm_stats_<기간>.json` 에 한 번 저장하고 팔끼리 같이 쓴다. 모델 버퍼로 들어가 체크포인트에 함께 저장된다.
- **층 정규화**는 각 TransformerConv 출력(128) 뒤 `LayerNorm` 이다. 배치 구성이 크게 흔들리는(배치 29% 세탁 0건) 우리 학습에서는 배치 통계를 쓰는 BatchNorm 보다 계좌마다 따로 정규화하는 LayerNorm 을 골랐다(FraudGT 와 같은 선택).

In [ ]:
VT = dict(zip(SCH['feature'], SCH['value_type']))                  # 스키마의 continuous / binary
def _norm_from(sums, sq, n, cols, strict=True):
    mean = sums / n; std = np.sqrt(np.maximum(sq / n - mean ** 2, 0.0))
    isbin = np.array([VT[c] == 'binary' for c in cols]); small = (~isbin) & (std < 1e-6)
    if strict:                                                        # 학습용 통계: 연속 열이 상수면 멈춘다
        assert not small.any(), f'연속 열인데 표준편차가 0: {[c for c, s in zip(cols, small) if s]}'
    fixed = isbin | small                                             # 자체 검사용(앞 행만): 상수 열은 (0,1) 로 두고 기록
    return dict(cols=cols, mean=np.where(fixed, 0.0, mean).tolist(), std=np.where(fixed, 1.0, std).tolist(), binary=isbin.tolist(),
                const_cols=[c for c, s in zip(cols, small) if s], raw_mean=mean.tolist(), raw_std=std.tolist(), rows=int(n))

def fit_input_norm(days, chunk=2_000_000, max_rows=None):
    """학습 날만으로 열별 평균·표준편차(float64 누적). max_rows 는 자체 검사용(앞 행만). 결과 json 을 팔끼리 같이 쓴다."""
    f = COMMON_FIX / f'input_norm_stats_{days[0]}_{days[-1]}.json'
    if max_rows is None and f.exists():
        return json.loads(f.read_text())
    out = {}
    for name, fld, cols in (('node', 'node_x', NODE19), ('edge', 'context_x', EDGE17)):
        s = np.zeros(len(cols)); s2 = np.zeros(len(cols)); n = 0
        for d in days:
            a = np.load(FC / f'day={d}' / f'{fld}.npy', mmap_mode='r'); m = len(a) if max_rows is None else min(len(a), max_rows)
            for st in range(0, m, chunk):
                b = np.asarray(a[st:min(st + chunk, m)], dtype=np.float64); s += b.sum(0); s2 += (b * b).sum(0); n += len(b)
        out[name] = _norm_from(s, s2, n, cols, strict=max_rows is None)
    out.update(days=[days[0], days[-1], len(days)], 정의='학습 날 캐시 전 행의 평균·모표준편차, 이진 열은 (0,1) 그대로', max_rows=max_rows)
    if max_rows is None:
        f.write_text(json.dumps(out, ensure_ascii=False, indent=1))
    return out

#### 후보 2 — 배치 손실 균형(A3)
- 원본의 `cross_entropy(weight=w)` 는 배치마다 **그 배치의 Σw** 로 나눈다. 그래서 세탁이 몰린 배치의 세탁 행은 몫이 작아진다(측정: 행마다 약 296배 차이, 중앙의 세탁 행은 정상 행의 약 4배 영향).
- 수정판은 **epoch 전체의 고정 분모** `W̄ = Σ(학습 행 가중치) ÷ 배치 수` 로 나눈다 → 어느 배치에서든 행 하나의 몫이 `w_i / W̄` 로 같다. 평균적인 손실 크기는 원본과 같다.
- 주의: 세탁이 수백~천여 건 몰린 배치는 손실이 커진다(최대 약 47배). 기울기 자르기(노름 5, 원본 그대로)가 이를 막는다.

In [ ]:
def make_loss(w, W_bar=None, fx=None):
    fx = fx or FIX
    if not fx['loss_balance']:
        return lambda logits, y: F.cross_entropy(logits, y, weight=w)                         # 원본: Σ w·l ÷ (배치 안 Σw)
    assert W_bar and W_bar > 0
    return lambda logits, y: (F.cross_entropy(logits, y, reduction='none') * w[y]).sum() / W_bar   # 수정: Σ w·l ÷ W̄(고정)

def epoch_weight_mass(days, B):
    """학습 행 수 · 배치 수 · W̄ (원본 학습 순서와 같은 행 집합)."""
    n_pat = n_norm = n_b = 0
    for d in days:
        tid = np.load(FC / f'day={d}' / 'target_ids.npy'); y = merge_labels(tid, np.load(FC / f'day={d}' / 'y9.npy'))
        pos = train_positions_ids(tid, y); p = int((y[pos] < 8).sum())
        n_pat += p; n_norm += len(pos) - p; n_b += -(-len(pos) // B)
    return dict(n_pattern=n_pat, n_normal=n_norm, batches=n_b, W_total=POS_W * n_pat + n_norm, W_bar=(POS_W * n_pat + n_norm) / n_b)

#### 후보 3 — 규칙적 섞기(A4) · **무작위 금지 규칙과 맞는지 사용자 판단 필요**
- 하루 안의 학습 대상(edge_id 순)을 `위치 = (i × k) mod n` 으로 재배열한다(k = 4099 부터 n 과 서로소가 될 때까지 +2).
- 난수가 없어 매번 같다.
- 날짜 순서와 하루 그래프(문맥)는 그대로라 **미래 정보가 들어가지 않는다**. 바뀌는 것은 같은 날 안에서 손실을 계산하는 순서뿐이다.
- 원인 분석(D3)에서 이 규칙으로 계산했을 때 세탁 0건 배치가 29% → 0.3% 로 줄었다.

In [ ]:
def interleave_k(n, k0=4099):
    k = k0
    while math.gcd(k, n) != 1:
        k += 2
    return k

def day_order(pos, fx=None, ep=0, d=None):
    fx = fx or FIX
    if fx['shuffle'] and len(pos) > 1:                               # 시드 셔플(09-29 사용자 승인): 태훈님 daily_pipeline.py 545줄과 같은 시드 식, ep 는 0부터
        return pos[np.random.default_rng(SEED + ep + pd.Timestamp(d).dayofyear).permutation(len(pos))]
    if not fx['interleave'] or len(pos) < 2:
        return pos                                                   # 원본: edge_id 순
    n = len(pos); k = interleave_k(n)
    return pos[(np.arange(n, dtype=np.int64) * k) % n]

#### 후보 4 — dropout · weight decay(A5) · **dropout 은 난수를 쓴다(시드 42 고정) — 사용자 판단 필요**
- 값은 FraudGT 설정(어텐션 dropout 0.2, AdamW weight decay 1e-5)과 Multi-GNN 헤드 dropout(0.1~0.29 중 0.1)이다. 조사 에이전트가 옮긴 값이고 원문은 다시 확인하지 않았다. 우리 데이터에서 튜닝하지 않은 임의값이다.
- weight decay 가 0 이면 원본과 같은 `Adam` 을 쓴다.

In [ ]:
def make_optimizer(model, fx=None):
    fx = fx or FIX
    if fx['weight_decay'] > 0:
        return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=fx['weight_decay'])
    return torch.optim.Adam(model.parameters(), lr=cfg.lr)           # 원본

### [B] 캐시를 더 만들어야 하는 것

#### 후보 5 — 역방향 메시지(+ ego) · 나가는 문맥 캐시 추가 필요
- **나가는 문맥**: 계좌마다 **보낸 거래 중 최근 10건**(30일 창, D일 포함 — 원본 문맥과 같은 창·같은 개수 규칙을 방향만 바꿈)
  - `/tmp/gnn36_out/day=…/out_context_{src,dst,x,ids}.npy` 로 **새로** 저장한다. 원본 캐시 `/tmp/gnn36_full` 은 건드리지 않는다.
  - 엣지 17열 = 원본 엣지 16열 + `amt_z_pay_ccy`(원본 캐시와 같은 방식)
- **자동 대조**(만들 때 날마다)
  - 같은 거래가 들어오는 문맥(원본 캐시)에도 있으면 17열 값이 비트 단위로 같아야 한다.
  - 첫날은 이 함수의 조회 방식으로 만든 **들어오는** 문맥이 원본 캐시와 거래 번호까지 같은지도 확인한다.
- **양방향 샘플러**: 시작 계좌(분류할 거래의 양 끝)에서 받은 거래 · 보낸 거래를 1홉, 그 이웃의 받은 · 보낸 거래를 2홉으로 모은다. 같은 거래가 두 목록에 다 있으면 한 번만 쓴다.
- **모델**: 층마다 `conv(받은 쪽, 원본) + conv_r(보낸 쪽, 뒤집은 엣지, 자기 값 없이)` 를 더한다(Egressy et al. 역방향 메시지의 합산형).
- **ego**: 배치 안 분류 대상 거래의 양 끝 계좌에 1, 나머지 0 인 열을 노드 입력에 붙인다(19 → 20열) — **입력 열 추가라 36피처 결정과 충돌하는지 사용자 판단 필요.**

In [ ]:
def load_out_context(cfg, day, direction='out'):
    """보낸 거래 최근 10건(direction='out') 또는 받은 거래 최근 10건(direction='in', 원본과 대조용). 원본 load_day 의 hist 창과 같은 기간."""
    require_stage(cfg, 'features'); day = pd.Timestamp(day).normalize(); start = day - pd.Timedelta(days=cfg.history_days - 1)
    from dataclasses import replace as _dc_replace
    c = connect(_dc_replace(cfg, memory_limit='10GB'), read_only=True)   # 14GB 로는 노트북 A 가 mem_guard 19.14 에 0.1 GiB 까지 갔다(09-28) — 10GB 에서 값 같음(09-27 기록)
    try:
        c.execute(f"""CREATE TEMP VIEW hist AS SELECT f.* FROM features f
           WHERE f.day>={sql_string(str(start.date()))}::DATE AND f.day<={sql_string(str(day.date()))}::DATE""")
        key = 'src' if direction == 'out' else 'dst'
        names = ','.join(EDGE_FEATURES)
        return c.execute(f"""SELECT edge_id,src,dst,{names} FROM hist
          QUALIFY row_number() OVER(PARTITION BY {key} ORDER BY ts DESC,edge_id DESC)<={cfg.incoming_neighbors}
          ORDER BY {key},ts DESC,edge_id DESC""").fetchdf()
    finally:
        c.close()

if RUN_OUT_CACHE:
    import shutil
    assert not heavy_jobs(), f'다른 무거운 작업이 돌고 있다 — 단독으로만 실행: {heavy_jobs()}'
    assert anon_gib() < 6, f'메모리 사용이 이미 크다(anon {anon_gib():.1f} GiB)'
    FC_OUT.mkdir(parents=True, exist_ok=True); rows = []
    ALL = DAYS['train'] + DAYS['val'] + DAYS['test']
    for i, d in enumerate(ALL):
        dd = FC_OUT / f'day={d}'
        if (dd / '_DONE').exists():
            continue
        a_ = anon_gib(); free_ = shutil.disk_usage('/tmp').free / 2**30
        assert a_ < 16, f'메모리 anon {a_:.1f} GiB ≥ 16 — 멈춤(날별 _DONE 이 있어 다시 돌리면 이어서 만든다)'
        assert free_ > 5, f'/tmp 여유 {free_:.1f} GiB ≤ 5 — 멈춤'
        dd.mkdir(exist_ok=True); t0 = time.time()
        o = load_out_context(cfg, d, 'out'); ids = o.edge_id.to_numpy(np.int64)
        az = np.asarray(AZ[ids]); assert np.isfinite(az).all(), 'amt_z NaN'
        ox = np.concatenate([o[EDGE_FEATURES].to_numpy(np.float32)[:, ESEL], az[:, None]], 1).astype(np.float32)
        osrc, odst = o.src.to_numpy(np.int64), o.dst.to_numpy(np.int64); del o
        assert np.all(osrc[:-1] <= osrc[1:]) and np.bincount(np.unique(osrc, return_inverse=True)[1]).max() <= cfg.incoming_neighbors
        cids = np.load(FC / f'day={d}' / 'context_ids.npy'); cx = np.load(FC / f'day={d}' / 'context_x.npy', mmap_mode='r')
        _, ia, ib = np.intersect1d(ids, cids, assume_unique=False, return_indices=True)
        assert np.array_equal(ox[ia], np.asarray(cx[ib])), f'{d}: 같은 거래의 17열 값이 원본 캐시와 다르다'
        if i == 0:                                                   # 첫날: 이 조회 방식의 "받은 거래" 문맥 = 원본 캐시
            chk = load_out_context(cfg, d, 'in')
            assert np.array_equal(chk.edge_id.to_numpy(np.int64), cids), f'{d}: 받은 쪽 문맥이 원본 캐시와 다르다'
            del chk
        for f_, v in dict(out_context_src=osrc, out_context_dst=odst, out_context_x=ox, out_context_ids=ids).items():
            np.save(dd / f'{f_}.npy', v)
        (dd / '_DONE').write_text(d)
        gc.collect(); rows.append(dict(day=d, out_edges=len(ids), overlap_with_in=len(ia), sec=round(time.time() - t0, 1), anon=round(anon_gib(), 1),
                                       tmp_free_gib=round(shutil.disk_usage('/tmp').free / 2**30, 1)))
        pd.DataFrame(rows).to_csv(COMMON_FIX / 'out_cache_log.csv', index=False); status_fix(f'OUT_CACHE {d} {i+1}/{len(ALL)} · {rows[-1]}')
    if all((FC_OUT / f'day={d}' / '_DONE').exists() for d in ALL):
        (FC_OUT / '_ALL_DONE').write_text(time.strftime('%F %T')); status_fix('OUT_CACHE 끝')

def load_out(d):
    dd = FC_OUT / f'day={d}'; assert (dd / '_DONE').exists(), f'나가는 문맥 캐시 없음: {d} — FIX_RUN_OUT_CACHE=1 을 먼저(단독) 실행'
    return {k: np.load(dd / f'out_context_{k}.npy') for k in ('src', 'dst', 'x', 'ids')}

class BiNeighborSampler(FixedNeighborSampler):
    """받은 거래(원본 문맥) + 보낸 거래(나가는 문맥)로 양방향 2홉. 같은 거래가 두 목록에 다 있으면 받은 쪽 것 하나만 쓴다."""
    def __init__(self, day, out, ctx_ids):
        super().__init__(day)
        self.out, self.cids = out, np.asarray(ctx_ids)
        assert len(self.cids) == len(day.context_src)
        self.osrc, self.ostart, self.ocount = np.unique(out['src'], return_index=True, return_counts=True)
        assert np.all(out['src'][:-1] <= out['src'][1:])

    def outgoing(self, nodes):
        nodes = np.unique(nodes); i = np.searchsorted(self.osrc, nodes); ok = i < len(self.osrc)
        ok[ok] = self.osrc[i[ok]] == nodes[ok]; i = i[ok]
        if not len(i): return np.empty(0, np.int64)
        counts = self.ocount[i]; starts = self.ostart[i]
        offsets = np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)
        return np.repeat(starts, counts) + offsets

    def batch(self, positions):
        d = self.day; o = self.out; positions = np.asarray(positions, dtype=np.int64)
        seeds = np.unique(np.r_[d.target_src[positions], d.target_dst[positions]])
        i1 = self.incoming(seeds); o1 = self.outgoing(seeds)                       # 1홉: 받은 · 보낸
        fr = np.unique(np.r_[d.context_src[i1], o['dst'][o1]])                     # 1홉 이웃
        i2 = self.incoming(fr); o2 = self.outgoing(fr)                             # 2홉: 이웃의 받은 · 보낸
        ci = np.union1d(i1, i2); co = np.union1d(o1, o2)
        co = co[~np.isin(o['ids'][co], self.cids[ci])]                             # 받은 쪽에 이미 있는 거래는 뺌
        es = np.r_[d.context_src[ci], o['src'][co]]; ed = np.r_[d.context_dst[ci], o['dst'][co]]
        ex = np.concatenate([np.asarray(d.context_x[ci]), o['x'][co]])
        nodes = np.unique(np.r_[seeds, es, ed])
        ni = np.searchsorted(d.node_ids, nodes)
        assert (ni < len(d.node_ids)).all() and np.array_equal(np.asarray(d.node_ids)[ni], nodes), 'Unknown node mapping'
        ei = np.stack([np.searchsorted(nodes, es), np.searchsorted(nodes, ed)])
        targets = np.stack([np.searchsorted(nodes, d.target_src[positions]), np.searchsorted(nodes, d.target_dst[positions])])
        return np.asarray(d.node_x[ni]), ei, ex, targets, np.asarray(d.target_x[positions])

### 수정 모델 `DailyGNN9Fix` — 스위치를 모두 끄면 원본 `DailyGNN9` 와 같다
- 원본과 같은 이름·같은 순서로 `conv1 → conv2 → head` 를 만든다. 그래서 **모두 끄면 초기 가중치와 체크포인트 형식까지 원본과 같다**(§10 ① 에서 epoch 3 가중치로 점수 차이 0 확인).
- 켠 것만 모듈이 추가된다: `ln1·ln2`(층 정규화), `conv1r·conv2r`(역방향), 정규화 버퍼, 헤드 앞 `Dropout`.

In [ ]:
class DailyGNN9Fix(torch.nn.Module):
    def __init__(self, fx=None, norm=None, n_node=19, n_edge=17, k=K):
        super().__init__()
        fx = dict(fx or FIX); self.fx = fx; H, heads = cfg.hidden, cfg.heads
        n_in = n_node + (1 if fx['ego'] else 0)
        pa = fx['dropout_attn']
        self.conv1 = TransformerConv(n_in, H // heads, heads=heads, edge_dim=n_edge, dropout=pa)
        self.conv2 = TransformerConv(H, H // heads, heads=heads, edge_dim=n_edge, dropout=pa)
        head = [torch.nn.Linear(2 * H + n_edge, H), torch.nn.ReLU(), torch.nn.Linear(H, k)]
        if fx['dropout_head'] > 0:
            head = [torch.nn.Dropout(fx['dropout_head'])] + head
        self.head = torch.nn.Sequential(*head)
        if fx['reverse_mp']:                                          # 보낸 쪽 집계(뒤집은 엣지). 자기 값은 원본 conv 에서 이미 더함
            self.conv1r = TransformerConv(n_in, H // heads, heads=heads, edge_dim=n_edge, dropout=pa, root_weight=False)
            self.conv2r = TransformerConv(H, H // heads, heads=heads, edge_dim=n_edge, dropout=pa, root_weight=False)
        if fx['layer_norm']:
            self.ln1, self.ln2 = torch.nn.LayerNorm(H), torch.nn.LayerNorm(H)
        if fx['input_norm']:
            assert norm is not None, '입력 표준화 통계(fit_input_norm)가 필요하다'
            t = lambda v: torch.tensor(v, dtype=torch.float32)
            self.register_buffer('in_nm', t(norm['node']['mean'])); self.register_buffer('in_ns', t(norm['node']['std']))
            self.register_buffer('in_em', t(norm['edge']['mean'])); self.register_buffer('in_es', t(norm['edge']['std']))

    def _layer(self, conv, convr, ln, x, ei, ea):
        h = conv(x, ei, ea)
        if convr is not None:
            h = h + convr(x, ei.flip(0), ea)                        # 뒤집은 엣지 = 받은 계좌 → 보낸 계좌로 메시지
        if ln is not None:
            h = ln(h)
        return torch.relu(h)

    def forward(self, x, ei, ea, targets, target_attr):
        if self.fx['input_norm']:
            x = (x - self.in_nm) / self.in_ns; ea = (ea - self.in_em) / self.in_es; target_attr = (target_attr - self.in_em) / self.in_es
        if self.fx['ego']:
            ego = torch.zeros(x.size(0), 1, dtype=x.dtype, device=x.device); ego[targets[0]] = 1.0; ego[targets[1]] = 1.0
            x = torch.cat([x, ego], 1)
        h = self._layer(self.conv1, getattr(self, 'conv1r', None), getattr(self, 'ln1', None), x, ei, ea)
        h = self._layer(self.conv2, getattr(self, 'conv2r', None), getattr(self, 'ln2', None), h, ei, ea)
        return self.head(torch.cat([h[targets[0]], h[targets[1]], target_attr], dim=1))

print('파라미터 수(팔별):', {a: sum(p.numel() for p in DailyGNN9Fix(dict(DEFAULTS, **v), norm=dict(node=dict(mean=[0.0]*19, std=[1.0]*19), edge=dict(mean=[0.0]*17, std=[1.0]*17)) if v.get('input_norm') else None).parameters()) for a, v in ARMS.items()})

## §8 학습 루프 · 손실 — `FIX_RUN_TRAIN` (원본 §8 과 같은 구조, 스위치만 반영)
- 원본과 같은 것: 날짜 순, CSSMC r300 학습 행, 가중치 300/1, lr 0.002, 배치 4,096, 시드 42(`FIX_SEED` 로 바꿈), 상한 15 · patience 3, epoch 마다 val macro-AP, 체크포인트(이어 돌기)
- 09-29 추가: 라벨 병합(`label_merge` — 패턴 밖 세탁을 정상 클래스로, train 43,095행 전부 가중치 1), 시드 셔플(`shuffle`), epoch 마다 val 바닥·조건부 8종과 학습 중 유형 CE 기록
- 스위치 반영: 샘플러(역방향), 하루 안 순서(규칙적 섞기), 손실(균형), 옵티마이저(weight decay), 모델(정규화 · dropout · ego)
- **안전장치**: 시작할 때 다른 무거운 작업이 돌면 멈춘다. 날마다 메모리가 18.5 GiB 를 넘으면 체크포인트를 남기고 멈춘다(mem_guard 19.1 보다 먼저).

In [ ]:
CS = np.load(CSSMC_PATH)
PO_RAW = (Y9_RAW == 8) & (IP_RAW == 1)                               # 패턴 밖 세탁(원본 행 번호) — 캐시 y9.npy 에서는 −1(뺌)
def merge_labels(tid, y, fx=None):
    """라벨 병합(`label_merge`, 09-29 사용자 결정 · 08-30 팀 결정): 패턴 밖 세탁을 정상 클래스(8 = API 의 0 NORMAL)로. 끄면 원본(−1 = 뺌)."""
    fx = fx or FIX
    if not fx['label_merge']:
        return y
    po = PO_RAW[tid]
    assert np.all(y[po] == -1), '패턴 밖 세탁인데 캐시 라벨이 −1 이 아니다'
    y = np.array(y, copy=True); y[po] = 8
    return y

if '_load_cached_raw' not in globals():
    _load_cached_raw = load_cached                                   # 원본 §4 load_cached(캐시 y9.npy 그대로)
def load_cached(d, root=FC):
    day, y = _load_cached_raw(d, root)
    return day, merge_labels(day.target_ids, y)

def train_positions_ids(tid, y):
    """학습 대상(edge_id 순서): 패턴 세탁 전부 + CSSMC 로 고른 정상 — 원본 §8 train_positions 와 같음.
    병합이면 패턴 밖 세탁(정상 클래스, 가중치 1)을 전부 더한다(CSSMC 는 진짜 정상만 골랐으므로 따로 넣는다. train 43,095행)."""
    q = np.minimum(np.searchsorted(CS, tid), len(CS) - 1)
    keep = ((y >= 0) & (y < 8)) | ((CS[q] == tid) & (y == 8))
    if FIX['label_merge']:
        keep |= PO_RAW[tid] & (y == 8)
    return np.flatnonzero(keep)

def po_keep_mask(split):
    """split_scores 가 채점한 행(y ≥ 0, 같은 순서) 중 패턴 밖 세탁 표시(병합일 때만 True 가 생김)."""
    out = []
    for d in DAYS[split]:
        tid = np.load(FC / f'day={d}' / 'target_ids.npy'); y = merge_labels(tid, np.load(FC / f'day={d}' / 'y9.npy'))
        out.append(PO_RAW[tid[y >= 0]])
    return np.concatenate(out)
def train_positions(day, y):
    return train_positions_ids(day.target_ids, y)

def make_sampler(day, d, fx=None):
    fx = fx or FIX
    if fx['reverse_mp']:
        return BiNeighborSampler(day, load_out(d), np.load(FC / f'day={d}' / 'context_ids.npy'))
    return FixedNeighborSampler(day)

def predict(model, day, which, dev, d):
    P = np.empty((len(which), K), np.float32); sampler = make_sampler(day, d); model.eval(); B = cfg.target_batch_size
    with torch.no_grad():
        for st in range(0, len(which), B):
            ix = which[st:st + B]; P[st:st + len(ix)] = torch.softmax(model(*tensor_batch(sampler.batch(ix), dev)), 1).float().cpu().numpy()
    return P

def split_scores(model, split, dev):
    Ps, ys, Po = [], [], []
    for d in DAYS[split]:
        day, y = load_cached(d); keep = np.flatnonzero(y >= 0); out_ = np.flatnonzero((y < 0) & (IP_RAW[day.target_ids] == 1))
        if FIX['shuffle'] and len(keep) > 1:                             # 학습처럼 섞인 배치로 채점(ego·revmp 는 배치 구성에 따라 점수가 달라짐) → 원래 행 순서로 되돌림
            perm = np.random.default_rng([SEED, 7919, pd.Timestamp(d).dayofyear]).permutation(len(keep))
            P_ = np.empty((len(keep), K), np.float32); P_[perm] = predict(model, day, keep[perm], dev, d); Ps.append(P_)
        else:
            Ps.append(predict(model, day, keep, dev, d))
        ys.append(y[keep])
        if len(out_): Po.append(predict(model, day, out_, dev, d))
        del day; gc.collect()
    return np.concatenate(Ps), np.concatenate(ys), (np.concatenate(Po) if Po else np.zeros((0, K), np.float32))

def ap_binary(s, pos):
    """tree9_lib.py 264-292줄 그대로(노트북 B 의 val macro-AP 와 같은 함수). sklearn average_precision_score 와 같은 정의."""
    npos = int(pos.sum())
    if npos == 0:
        return float('nan')
    smin = s[pos].min(); m = s >= smin
    if int(m.sum()) < len(s):
        s, pos = s[m], pos[m]
    del m
    o = np.argsort(s, kind='stable'); ss = s[o][::-1]; yp = pos[o][::-1]; del o
    tp = np.cumsum(yp, dtype=np.int64); del yp
    last = np.empty(len(ss), bool); last[:-1] = ss[1:] != ss[:-1]; last[-1] = True; del ss
    tp_l = tp[last]; del tp
    n_l = np.flatnonzero(last) + 1; del last
    d = np.diff(tp_l, prepend=0); keep = d > 0
    return float(np.sum(d[keep] * (tp_l[keep] / n_l[keep])) / npos)

def macro_ap(P, y):
    aps = [ap_binary(P[:, k].astype(np.float64), y == k) for k in range(K)]; return float(np.nanmean(aps)), aps

def meta_of(norm, wm):
    return dict(arm=FIX_ARM, smoke=SMOKE, fix=FIX, node_features=NODE19, edge_features=EDGE17, classes=CLASS9, cfg=asdict(cfg),
                cssmc=str(CSSMC_PATH), cssmc_sha1=CSSMC_SHA1, input_norm=norm, weight_mass=wm, days={s: [v[0], v[-1], len(v)] for s, v in DAYS.items()},
                max_ep=MAX_EP, patience=PAT, seed=SEED, notebook_sha1=NB_SHA1, source='TGNN9클래스_수정판_한파일_20260929.ipynb')

if RUN_TRAIN:
    assert not heavy_jobs(), f'다른 무거운 작업이 돌고 있다: {heavy_jobs()}'
    assert _sha1(CSSMC_PATH) == CSSMC_SHA1, 'cssmc_r300.npy 가 09-28 기록과 다르다'
    if FIX['reverse_mp']:
        miss = [d for v in DAYS.values() for d in v if not (FC_OUT / f'day={d}' / '_DONE').exists()]
        assert not miss, f'나가는 문맥 캐시 없는 날 {miss[:3]} … — FIX_RUN_OUT_CACHE=1 먼저'
    DEV = 'cuda'; assert torch.cuda.is_available(); B = cfg.target_batch_size
    NORM = fit_input_norm(DAYS['train']) if FIX['input_norm'] else None
    WM = epoch_weight_mass(DAYS['train'], B) if FIX['loss_balance'] else None
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = DailyGNN9Fix(FIX, NORM).to(DEV); opt = make_optimizer(model)
    w = torch.full((K,), POS_W, device=DEV); w[8] = 1.0                    # 패턴 300 · 정상 1
    loss_fn = make_loss(w, WM['W_bar'] if WM else None)
    status_fix(f'학습 시작 · 파라미터 {sum(p.numel() for p in model.parameters()):,} · W̄ {WM["W_bar"] if WM else "-"} · 정규화 {bool(NORM)}')
    ck = [] if SMOKE else sorted((OUT / 'models').glob('ep*.pt'), key=lambda p: int(p.stem[2:]))   # 스모크는 이어 돌지 않는다(고친 학습 코드를 매번 검사)
    best, best_ep, stale, hist, ep0 = -1.0, -1, 0, [], 0
    if ck:                                                              # 이어서 돌기
        s = torch.load(ck[-1], map_location=DEV, weights_only=False)
        assert s['meta']['fix'] == FIX and s['meta'].get('seed', 42) == SEED, f"체크포인트의 스위치·시드가 지금과 다르다: {s['meta']['fix']} · {s['meta'].get('seed')}"
        if s['meta'].get('notebook_sha1') not in (None, NB_SHA1): print('주의: 체크포인트를 만든 노트북 지문이 지금과 다르다', s['meta'].get('notebook_sha1'), NB_SHA1)
        model.load_state_dict(s['model']); opt.load_state_dict(s['opt']); best, best_ep, stale, hist, ep0 = s['best'], s['best_ep'], s['stale'], s['hist'], s['epoch']
        status_fix(f'체크포인트 ep{ep0} 에서 이어서')
    for ep in range(ep0, MAX_EP):
        if stale >= PAT: break
        t0 = time.time(); n_tr = nb = 0; lsum = 0.0; torch.cuda.reset_peak_memory_stats()
        tce = torch.zeros((), device=DEV, dtype=torch.float64); tacc = torch.zeros((), device=DEV, dtype=torch.long); tn = 0
        for d in DAYS['train']:
            a = anon_gib()
            if a > 18.5:
                raise RuntimeError(f'메모리 anon {a:.1f} GiB > 18.5 — 멈춤(체크포인트 ep{ep} 까지 있음)')
            day, y = load_cached(d); pos = day_order(train_positions(day, y), ep=ep, d=d); sampler = make_sampler(day, d); model.train()
            for st in range(0, len(pos), B):
                ix = pos[st:st + B]; opt.zero_grad(set_to_none=True)
                logits = model(*tensor_batch(sampler.batch(ix), DEV)); yt = torch.as_tensor(y[ix], device=DEV, dtype=torch.long)
                loss = loss_fn(logits, yt)
                if not torch.isfinite(loss): raise ValueError('Nonfinite loss')
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5., error_if_nonfinite=True); opt.step()
                lsum += float(loss.detach()); nb += 1
                with torch.no_grad():                                   # 학습 중 유형 추적(진단 G 후속): 세탁 행의 조건부 8종 CE · 정답(순전파 값)
                    lm = yt < 8; lg = logits.detach()[lm][:, :8].float(); yy = yt[lm]
                    tce += (torch.logsumexp(lg, 1) - lg.gather(1, yy[:, None]).squeeze(1)).sum().double(); tacc += (lg.argmax(1) == yy).sum()
                tn += int((y[ix] < 8).sum())
            n_tr += len(pos); del day, sampler; gc.collect()
        t1 = time.time(); Pv, yv, _ = split_scores(model, 'val', DEV); m, aps = macro_ap(Pv, yv)
        lv = yv < 8; det = ap_binary((1.0 - Pv[:, 8]).astype(np.float64), lv); floor = (aps[8] + det) / 9          # 유형 정보 없음 바닥(유형붕괴 보고서 §0-4)
        cond = float((Pv[lv][:, :8].argmax(1) == yv[lv]).mean()); t2 = time.time()
        hist.append(dict(epoch=ep + 1, train_rows=n_tr, batches=nb, loss_mean=lsum / max(nb, 1), train_sec=round(t1 - t0), val_sec=round(t2 - t1),
                         val_macro_ap=m, val_세탁여부_ap=det, val_바닥=floor, val_바닥대비=m - floor, val_조건부8종=cond, seed=SEED, 라벨='병합' if FIX['label_merge'] else '제거',
                         train_유형CE=float(tce) / max(tn, 1), train_조건부8종=int(tacc) / max(tn, 1), train_세탁행=tn,
                         **{f'ap_{CLASS9[k]}': aps[k] for k in range(K)},
                         gpu_peak_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2), anon=round(anon_gib(), 1)))
        if m > best: best, best_ep, stale = m, ep + 1, 0
        else: stale += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep + 1, best=best, best_ep=best_ep, stale=stale, hist=hist, meta=meta_of(NORM, WM)),
                   OUT / 'models' / f'ep{ep+1}.pt')
        pd.DataFrame(hist).to_csv(OUT / 'results/학습기록.csv', index=False, encoding='utf-8-sig')
        status_fix(f'epoch {ep+1}/{MAX_EP} · val macro-AP {m:.5f}(최고 ep{best_ep} {best:.5f}) · 바닥 {floor:.4f}(대비 {m - floor:+.4f}) · 조건부8종 val {cond:.3f}/train {hist[-1]["train_조건부8종"]:.3f} · train 유형CE {hist[-1]["train_유형CE"]:.3f} · 학습 {t1-t0:.0f}초 · 검증 {t2-t1:.0f}초 · GPU {hist[-1]["gpu_peak_gib"]} GiB')
        del Pv, yv; gc.collect()

## §9 채점 — 규칙별 epoch 선택 → val·test 한 번 (원본 §9 + 부수 지표) — `FIX_RUN_EVAL`

**팔 비교는 val 로만 한다**(누수·재현성 검토 P5b, 09-29 사용자 승인). test 는 계산해 `results/지표.csv` 와 행별 확률 파일에만 저장하고, 화면 표·상태 줄에는 띄우지 않는다. test 는 팀이 val 로 최종 설정을 고른 뒤 그 팔 하나만 연다(여러 팔의 test 를 보고 고르면 고른 팔의 test 점수가 부풀고, test 가 선택에 쓰인 셈이 된다).

In [ ]:
def rule_pick(vals, max_ep, pat):
    b, be, s = -1.0, -1, 0
    for i, v in enumerate(vals[:max_ep]):
        if v > b: b, be, s = v, i + 1, 0
        else:
            s += 1
            if s >= pat: break
    return be

if RUN_EVAL:
    from sklearn.metrics import average_precision_score
    DEV = 'cuda'; H = pd.read_csv(OUT / 'results/학습기록.csv'); v = H['val_macro_ap'].tolist()
    PICK = {r: rule_pick(v, *mp) for r, mp in RULES.items()} if not SMOKE else {r: rule_pick(v, MAX_EP, PAT) for r in RULES}
    NORM = fit_input_norm(DAYS['train']) if FIX['input_norm'] else None
    model = DailyGNN9Fix(FIX, NORM).to(DEV); out, done_e = [], {}
    for rule, e in PICK.items():
        if e in done_e:
            out += [dict(o, rule=rule) for o in done_e[e]]; continue
        model.load_state_dict(torch.load(OUT / 'models' / f'ep{e}.pt', map_location=DEV, weights_only=False)['model']); n0 = len(out)
        for sp in ('val', 'test'):
            status_fix(f'채점 {rule} ep{e} {sp}')
            P, y, Po = split_scores(model, sp, DEV)
            np.save(SC / f'ep{e}_{sp}_P.npy', P); np.save(SC / f'ep{e}_{sp}_y.npy', y); np.save(SC / f'ep{e}_{sp}_Po.npy', Po)
            pred = P.argmax(1).astype(np.int8); pos = y < 8; pp = pred != 8; det_ap = float(average_precision_score(pos, 1.0 - P[:, 8]))
            po_k = po_keep_mask(sp) if FIX['label_merge'] else np.zeros(len(y), bool); assert len(po_k) == len(y)
            aps = [float(average_precision_score(y == k, P[:, k])) if (y == k).any() else float('nan') for k in range(K)]
            b_tp = int((pos & pp).sum()); b_p = b_tp / max(int(pp.sum()), 1); b_r = b_tp / max(int(pos.sum()), 1)
            po = Po.argmax(1) if len(Po) else np.array([], np.int64)
            if po_k.any():                                                  # 병합 팔: 패턴 밖 행을 뺀(앞 팔과 같은 행) macro-AP · 바닥대비
                q_ = ~po_k; aps_q = [float(average_precision_score(y[q_] == k, P[q_, k])) for k in range(K)]
                det_q = float(average_precision_score(pos[q_], 1.0 - P[q_, 8])); mq = float(np.nanmean(aps_q)); flq = (aps_q[8] + det_q) / 9
            else:
                mq = flq = float('nan')
            out.append(dict(arm=FIX_ARM, run=RUN_NAME, seed=SEED, 라벨='병합' if FIX['label_merge'] else '제거', rule=rule, split=sp, selected_epoch=e, rows=len(y), macro_AP_9=float(np.nanmean(aps)),
                            macro_AP_9_패턴밖제외=mq, 바닥대비_패턴밖제외=mq - flq,
                            macro_AP_패턴8종=float(np.nanmean(aps[:8])), AP_정상=aps[8], 세탁여부_AP=det_ap, 바닥_macroAP=(aps[8] + det_ap) / 9, 바닥대비=float(np.nanmean(aps)) - (aps[8] + det_ap) / 9,
                            세탁여부_argmax_정밀도=b_p, 세탁여부_argmax_재현율=b_r, 세탁으로_예측한_행=int(pp.sum()),
                            패턴행중_유형정답률=float((pred[pos] == y[pos]).mean()),
                            조건부_8종_정답률=float((P[pos][:, :8].argmax(1) == y[pos]).mean()),          # 부수 지표: 정답 패턴 행에서 8개 유형만 비교
                            뺀_패턴밖세탁_행=len(po), 패턴밖세탁중_세탁으로_예측=float((po != 8).mean()) if len(po) else float('nan'),
                            병합_패턴밖세탁_행=int(po_k.sum()), 병합_패턴밖세탁중_패턴으로_예측=float((pred[po_k] != 8).mean()) if po_k.any() else float('nan')))
            del P, Po; gc.collect()
        done_e[e] = out[n0:]
    M = pd.DataFrame(out)
    if not SMOKE:                                                   # 병합 = 원본 + 패턴 밖 세탁 val 16,691 · test 17,063(09-29 캐시에서 셈)
        EXP_ROWS = {'val': 33_649_784, 'test': 30_407_864} if FIX['label_merge'] else {'val': 33_633_093, 'test': 30_390_801}
        assert all(set(M.query(f"split=='{s_}'")['rows']) == {n_} for s_, n_ in EXP_ROWS.items()), f'평가 행 수가 기대와 다르다 {EXP_ROWS}'
    M.to_csv(OUT / 'results/지표.csv', index=False, encoding='utf-8-sig')
    base = pd.read_csv(W / 'results/B3_지표.csv').query("rule=='규칙15_3'").assign(arm='원본(B)')
    base = base.assign(바닥_macroAP=(base['AP_정상'] + base['세탁여부_AP']) / 9); base = base.assign(바닥대비=base['macro_AP_9'] - base['바닥_macroAP'])
    cols = ['arm', 'split', 'selected_epoch', 'macro_AP_9', '바닥_macroAP', '바닥대비', 'macro_AP_9_패턴밖제외', '바닥대비_패턴밖제외', 'macro_AP_패턴8종', '세탁여부_AP', '조건부_8종_정답률', '세탁여부_argmax_정밀도', '세탁여부_argmax_재현율', '패턴행중_유형정답률']
    # 팔 비교는 val 로만(누수·재현성 검토 P5b) — test 값은 지표.csv 에만 있고 여기서는 보이지 않는다
    print(pd.concat([base.query("split=='val'").reindex(columns=cols), M.query("rule=='규칙15_3' and split=='val'")[cols]]).round(4).to_string(index=False))
    if FIX['label_merge']:
        print('주의: 원본(B) 줄은 병합 전 라벨(패턴 밖 세탁 뺌)이다 — macro-AP 는 직접 비교하지 않고 바닥대비 · 조건부 8종으로 본다')
    print('test 지표는 results/지표.csv 에만 저장했다 — 팔 비교·선택에 쓰지 않는다(최종 팔 하나만 연다)')
    _V = M[M['split'] == 'val'].drop_duplicates('rule').set_index('rule')['macro_AP_9']
    _F = M[M['split'] == 'val'].drop_duplicates('rule').set_index('rule')['바닥대비']
    status_fix('DONE 채점 · 고른 epoch ' + str(PICK) + ' · val macro-AP(9) ' + ', '.join(f'{r}: {_V[r]:.4f}(바닥대비 {_F[r]:+.4f})' for r in PICK) + ' · test 는 지표.csv 에만')

## §10 자체 검사 (`FIX_RUN_SELFCHECK`, CPU 몇 분, 캐시를 mmap 으로 읽음 — 학습 중인 다른 작업과 같이 돌아도 메모리 약 1~2 GiB)
1. **모두 끔 = 원본**: `DailyGNN9Fix(모두 끔)` 과 원본 `DailyGNN9` 에 학습된 epoch 3 가중치를 넣고 같은 배치(첫 train 날 1,024건) → 9개 점수 차이 0
2. [A] 팔마다 앞뒤 계산(CPU, 256건): 손실·기울기가 유한한가, 파라미터 수
3. 규칙적 섞기: 순서가 전단사(빠짐·중복 없음)이고 매번 같은가, 그날 세탁 0건 배치 비율 전후
4. 손실 균형: 세탁이 적은 배치와 몰린 배치에서 세탁 행 하나의 기울기가 같은가(원본은 다름)
5. [B] 양방향 샘플러: 손으로 만든 작은 그래프에서 모은 거래·계좌가 기대와 같은가, 역방향 모델이 도는가(실제 나가는 문맥 캐시는 아직 없어 합성으로 대신)
6. 입력 표준화 통계 함수: 앞 20만 행으로 경로 확인(이진 열은 그대로)

In [ ]:
if RUN_SELFCHECK:
    import importlib
    sys.path.insert(0, str(CODE_FILE.parent)); dp = importlib.import_module('daily_pipeline')
    from torch_geometric.nn import TransformerConv as LibTransformerConv
    d0 = DAYS['train'][0]; dd = FC / f'day={d0}'
    day = DayData(day=d0, **{f: np.load(dd / f'{f}.npy', mmap_mode='r') for f in CACHE_FIELDS}); y = np.load(dd / 'y9.npy')
    pos_all = train_positions(day, y); pos = pos_all[:1024]
    batch = FixedNeighborSampler(day).batch(pos); tb = tensor_batch(batch, 'cpu')
    assert all(np.array_equal(np.asarray(u), np.asarray(v)) for u, v in zip(batch, dp.FixedNeighborSampler(day).batch(pos))), '샘플러가 원본과 다르다'
    state = torch.load(W / 'models/gnn36_cssmc_r300/ep3.pt', map_location='cpu', weights_only=False)['model']
    m0, m1, m2 = DailyGNN9(), DailyGNN9Fix(DEFAULTS), DailyGNN9(conv=LibTransformerConv)
    for m in (m0, m1, m2): m.load_state_dict(state); m.eval()
    with torch.no_grad():
        o0, o1, o2 = m0(*tb), m1(*tb), m2(*tb)
    d01, d02 = float((o0 - o1).abs().max()), float((o0 - o2).abs().max())
    print(f'① 모두 끔 vs 원본 DailyGNN9: 최대 차이 {d01} · (참고) 원본 vs 라이브러리 층: {d02}')
    assert d01 == 0.0 and d02 < 1e-5

    qn = fit_input_norm([d0], max_rows=200_000)                     # ⑥ 겸 ② 용(저장 안 함)
    assert all(qn['node']['std'][j] == 1.0 and qn['node']['mean'][j] == 0.0 for j, b in enumerate(qn['node']['binary']) if b)
    print('⑥ 입력 표준화(앞 20만 행): 노드 연속 열 표준편차', np.round(np.array(qn['node']['raw_std'])[~np.array(qn['node']['binary'])], 3).tolist(), '· 앞 행에서 상수라 (0,1) 로 둔 열', qn['node']['const_cols'] + qn['edge']['const_cols'])

    w_cpu = torch.full((K,), POS_W); w_cpu[8] = 1.0
    b256 = tensor_batch(FixedNeighborSampler(day).batch(pos_all[:256]), 'cpu'); y256 = torch.as_tensor(y[pos_all[:256]], dtype=torch.long)
    rows = []
    for arm, v in ARMS.items():
        fx = dict(DEFAULTS, **v)
        if fx['reverse_mp']:
            continue                                                  # [B] 는 ⑤ 에서 합성 그래프로
        torch.manual_seed(42); m = DailyGNN9Fix(fx, qn if fx['input_norm'] else None); m.train()
        lf = make_loss(w_cpu, 8116.0, fx)                             # W̄ 는 자리값(실제 값은 학습 때 계산)
        loss = lf(m(*b256), y256); loss.backward()
        g = [p.grad for p in m.parameters() if p.grad is not None]
        rows.append(dict(팔=arm, 손실=round(float(loss), 4), 기울기_유한=bool(all(torch.isfinite(t).all() for t in g)), 파라미터=sum(p.numel() for p in m.parameters()),
                         옵티마이저=type(make_optimizer(m, fx)).__name__))
    R2 = pd.DataFrame(rows); print('② [A] 팔별 앞뒤 계산'); print(R2.to_string(index=False)); assert R2['기울기_유한'].all()

    fxi = dict(DEFAULTS, interleave=True); o_a = day_order(pos_all, fxi); o_b = day_order(pos_all, fxi)
    assert np.array_equal(o_a, o_b) and np.array_equal(np.sort(o_a), np.sort(pos_all)), '규칙적 섞기가 전단사가 아니거나 매번 다르다'
    zb = lambda p: float(np.mean([(y[p[s:s + 4096]] < 8).sum() == 0 for s in range(0, len(p), 4096)]))
    print(f'③ 규칙적 섞기(k={interleave_k(len(pos_all))}): 그날({d0}) 세탁 0건 배치 비율 {zb(pos_all):.3f} → {zb(o_a):.3f}')
    fxs = dict(DEFAULTS, shuffle=True); s_a, s_b, s_c = day_order(pos_all, fxs, 0, d0), day_order(pos_all, fxs, 0, d0), day_order(pos_all, fxs, 1, d0)
    assert np.array_equal(s_a, s_b) and np.array_equal(np.sort(s_a), np.sort(pos_all)) and not np.array_equal(s_a, s_c), '시드 셔플이 재현되지 않거나 전단사가 아니다'
    print(f'③-2 시드 셔플: 같은 epoch 은 같은 순서 · epoch 이 바뀌면 다른 순서 · 세탁 0건 배치 비율 {zb(pos_all):.3f} → {zb(s_a):.3f}')
    fxm = dict(DEFAULTS, label_merge=True); ym = merge_labels(np.asarray(day.target_ids), y, fxm); npo = int(PO_RAW[np.asarray(day.target_ids)].sum())
    assert int((ym == 8).sum()) == int((y == 8).sum()) + npo and np.array_equal(ym[ym != 8], y[ym != 8]) and np.array_equal(merge_labels(np.asarray(day.target_ids), y, DEFAULTS), y)
    print(f'③-3 라벨 병합: 그날({d0}) 패턴 밖 세탁 {npo}행 −1 → 정상 클래스(8), 다른 행은 그대로')

    def grad_of_one_pattern(lossf, n_pat):
        yb = torch.full((4096,), 8, dtype=torch.long); yb[:n_pat] = 0
        z = torch.zeros(4096, K, requires_grad=True); lossf(z, yb).backward(); return float(z.grad[0].abs().sum())
    lm, lb = make_loss(w_cpu, None, DEFAULTS), make_loss(w_cpu, 8116.0, dict(DEFAULTS, loss_balance=True))
    g_m = (grad_of_one_pattern(lm, 1), grad_of_one_pattern(lm, 1000)); g_b = (grad_of_one_pattern(lb, 1), grad_of_one_pattern(lb, 1000))
    print(f'④ 세탁 행 하나의 기울기 크기(세탁 1건 배치 vs 1,000건 배치): 원본 {g_m[0]:.5f} vs {g_m[1]:.5f}(비 {g_m[0]/g_m[1]:.0f}배) · 균형 {g_b[0]:.5f} vs {g_b[1]:.5f}')
    assert abs(g_b[0] - g_b[1]) < 1e-9 and g_m[0] / g_m[1] > 10

    # ⑤ 합성 그래프: 계좌 0~5, 분류할 거래 = 20(0→4). 받은 문맥(받는 계좌 순): 10(1→0) 11(2→0) 12(3→1) 20(0→4). 나가는 문맥(보낸 계좌 순): 20(0→4) 21(0→5) 12(3→1) 22(4→5)
    ex17 = lambda ids: np.array([[i] * 17 for i in ids], np.float32)
    syn = DayData(day='합성', node_ids=np.arange(6), node_x=np.tile(np.arange(6, dtype=np.float32)[:, None], (1, 19)),
                  context_src=np.array([1, 2, 3, 0]), context_dst=np.array([0, 0, 1, 4]), context_x=ex17([10, 11, 12, 20]),
                  target_ids=np.array([20]), target_src=np.array([0]), target_dst=np.array([4]), target_x=ex17([20]), labels=None)
    out_syn = dict(src=np.array([0, 0, 3, 4]), dst=np.array([4, 5, 1, 5]), x=ex17([20, 21, 12, 22]), ids=np.array([20, 21, 12, 22]))
    nx_, ei_, ea_, tg_, ta_ = BiNeighborSampler(syn, out_syn, np.array([10, 11, 12, 20])).batch([0])
    got = sorted(ea_[:, 0].astype(int).tolist()); _, ei0, ea0, _, _ = FixedNeighborSampler(syn).batch([0])
    print('⑤ 양방향 샘플러 거래', got, '(원본 받은 쪽만:', sorted(ea0[:, 0].astype(int).tolist()), ') · 계좌 수', len(nx_))
    assert got == [10, 11, 12, 20, 21, 22] and len(nx_) == 6 and sorted(ea0[:, 0].astype(int).tolist()) == [10, 11, 12, 20]
    for arm in ('revmp', 'revmp_ego'):
        torch.manual_seed(42); m = DailyGNN9Fix(dict(DEFAULTS, **ARMS[arm])); out_ = m(*tensor_batch((nx_, ei_, ea_, tg_, ta_), 'cpu'))
        assert out_.shape == (1, K) and torch.isfinite(out_).all()
    print('⑤ 역방향 모델(revmp · revmp_ego) 합성 배치 통과')
    print('자체 검사 통과 — 모두 끄면 원본과 같고, 후보 5개 코드가 CPU 에서 돈다(실제 학습 효과는 미측정)')

## §11 API 클래스 순서로 바꾸기 (팀장 피드백 3 · `scores.parquet` 쓰기 전에)
- 이 노트북의 순서: 0 FAN-OUT · 1 FAN-IN · 2 CYCLE · 3 SCATTER-GATHER · 4 GATHER-SCATTER · 5 BIPARTITE · 6 STACK · 7 RANDOM · 8 정상(병합 팔에서는 정상 + 패턴 밖 = 패턴아님)
- API 순서: 0 NORMAL · 1 FAN-OUT · 2 FAN-IN · 3 G-SCATTER · 4 S-GATHER · 5 CYCLE · 6 RANDOM · 7 BIPARTITE · 8 STACK
- 번호가 아니라 **이름으로** 맞춘다. 단순히 +1 하면 CYCLE·G-SCATTER·S-GATHER·RANDOM·BIPARTITE·STACK 이 뒤바뀐다.
- G-SCATTER = GATHER-SCATTER, S-GATHER = SCATTER-GATHER 로 봤다(팀 확인 필요). LightGBM 점수도 같은 함수로 바꾼다.
- `scores.parquet` 의 열 이름·형식은 API 명세를 확인한 뒤 쓴다(여기서는 쓰지 않는다).

In [ ]:
API_CLASSES = ['NORMAL', 'FAN-OUT', 'FAN-IN', 'G-SCATTER', 'S-GATHER', 'CYCLE', 'RANDOM', 'BIPARTITE', 'STACK']
API_TO_OURS = {'NORMAL': '정상', 'FAN-OUT': 'FAN-OUT', 'FAN-IN': 'FAN-IN', 'G-SCATTER': 'GATHER-SCATTER', 'S-GATHER': 'SCATTER-GATHER',
               'CYCLE': 'CYCLE', 'RANDOM': 'RANDOM', 'BIPARTITE': 'BIPARTITE', 'STACK': 'STACK'}
API_IDX = [CLASS9.index(API_TO_OURS[a]) for a in API_CLASSES]       # = [8, 0, 1, 4, 3, 2, 7, 5, 6]
assert API_IDX == [8, 0, 1, 4, 3, 2, 7, 5, 6] and sorted(API_IDX) == list(range(9))

def to_api_order(P):
    """행별 9클래스 확률(이 노트북 순서) → API 순서. 열 j = API_CLASSES[j]."""
    return np.asarray(P)[:, API_IDX]

def to_api_label(y):
    """이 노트북 라벨(0~8) → API 코드(0~8)."""
    inv = np.empty(9, np.int8); inv[API_IDX] = np.arange(9)
    return inv[np.asarray(y)]

_p = np.eye(9, dtype=np.float32)                                      # 자기 확인: 클래스 k 확률 1 → API 이름이 같은 자리로
assert all(API_TO_OURS[API_CLASSES[int(to_api_order(_p[k:k+1]).argmax())]] == CLASS9[k] for k in range(9))
assert all(API_TO_OURS[API_CLASSES[int(to_api_label([k])[0])]] == CLASS9[k] for k in range(9))
print('API 순서 변환 확인:', dict(zip(API_CLASSES, [CLASS9[i] for i in API_IDX])))